## Sommaire du pipeline Data Engineering

| Étape | Description | Table(s) produite(s) |
| --- | --- | --- |
| 1 | Fonctions utilitaires (nettoyage, suppression colonnes, sauvegarde) | — |
| 2 | Nettoyage DLV Livraisons | `dlv_deliveries_f_clean` |
| 3 | Nettoyage QIB Installations & Pannes | `qib_installations_breakdowns_monthly_kpi` |
| 4 | Nettoyage PRD Produits | `prd_products_dim_clean` |
| 5 | Nettoyage STC Sites (dimension) | `stc_sites_dim_clean` |
| 6 | Ingestion CSV EcoAct (risques climatiques) | `ecoact_data_clean` |
| 7 | KPIs de production annuelle (QYP) | `qyp_quarries_yearly_reporting_clean`, `qyp_quarries_yearly_kpi` |
| 8 | KPIs de réserves (RYP) | `ryp_quarries_reserves_yearly_reporting_clean`, `ryp_quarries_reserves_yearly_kpi` |
| 9 | Agrégation CA annuel par carrière | `ca_annuel_par_carriere` |
| 10 | Fiche carrière BI (jointure multi-sources) | `fiche_carriere_bi` |
| 11 | Scoring multicritères des carrières | `carriere_score_bi` |
| 12 | Analyse du mix produit | `product_mix_bi` |
| 13 | Risques climatiques détaillés (EcoAct → BI) | `climate_risks_bi`, `climate_risks_site_summary_bi` (V1) |
| 14 | Conversion multi-devises EUR | `fx_rates_to_eur`, `fiche_carriere_bi_eur` |
| 15 | Corrections qualité données site | `fiche_carriere_bi_eur` (overwrite) |
| 16 | Table résumée par carrière (vue 360°) | `carriere_summary_bi` |
| 17 | Résumé climatique V2 (score composite) | `climate_risks_site_summary_bi` (overwrite) |

# Objectif du pipeline Data Engineering

Ce notebook a pour objectif de construire une base de données analytique centralisée permettant :

- l’analyse stratégique des carrières Colas,
- le suivi des performances opérationnelles,
- l’étude des risques climatiques,
- ainsi que l’alimentation des dashboards Power BI et des analyses business.

Le pipeline repose sur plusieurs sources de données :
- données opérationnelles issues du DataLake Colas,
- données de production et de réserves,
- données de pannes des installations,
- données climatiques externes (EcoAct),
- données géographiques et structurelles des sites.

L’ensemble des transformations est réalisé avec PySpark sur Databricks afin de garantir :
- la scalabilité,
- la reproductibilité,
- la qualité des données,
- et la création de tables analytiques prêtes à être exploitées.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, NumericType, FloatType, DoubleType
from pyspark.sql.window import Window

# ============================================================
# CONFIG GLOBALE
# ============================================================

TEAM_SCHEMA = "efrei_estp_2026.team6"

EMPTY_VALUES = ["", "-", "(VIDE)", "(EMPTY)", "null", "NULL", "[BLANK]", "N/A", "NA", "nan", "NaN"]


# ============================================================
# FONCTIONS RÉUTILISABLES
# ============================================================

def clean_strings(df, empty_values=None):
    """
    Remplace les chaînes vides ou assimilées par None dans toutes les colonnes string.
    """
    if empty_values is None:
        empty_values = EMPTY_VALUES

    string_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, StringType)
    ]

    for c in string_cols:
        df = df.withColumn(
            c,
            F.when(F.trim(F.col(c)).isin(empty_values), F.lit(None))
             .otherwise(F.trim(F.col(c)))
        )

    return df


def drop_useless_columns(df, protected_cols=None, null_threshold=0.80, zero_threshold=0.95):
    """
    Supprime les colonnes avec :
    - plus de null_threshold (80%) de NULL/NaN
    - plus de zero_threshold (95%) de zéros (colonnes numériques)
    - une seule valeur distincte (constantes)

    Retourne (df_nettoyé, liste_colonnes_supprimées).
    """
    if protected_cols is None:
        protected_cols = []

    row_count = df.count()
    if row_count == 0:
        return df, []

    numeric_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, NumericType)
    ]

    nan_compatible_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, (FloatType, DoubleType))
    ]

    stats_exprs = []
    for c in df.columns:
        missing_cond = (
            F.col(c).isNull() | F.isnan(F.col(c))
            if c in nan_compatible_cols
            else F.col(c).isNull()
        )
        stats_exprs.append(
            F.sum(F.when(missing_cond, 1).otherwise(0)).alias(f"{c}__missing")
        )
        stats_exprs.append(
            F.countDistinct(F.col(c)).alias(f"{c}__distinct")
        )
        if c in numeric_cols:
            stats_exprs.append(
                F.sum(F.when(F.col(c) == 0, 1).otherwise(0)).alias(f"{c}__zero")
            )

    stats = df.agg(*stats_exprs).collect()[0].asDict()

    cols_to_drop = []
    for c in df.columns:
        if c in protected_cols:
            continue

        missing_rate = stats[f"{c}__missing"] / row_count
        distinct_count = stats[f"{c}__distinct"]

        if missing_rate > null_threshold:
            cols_to_drop.append(c)
        elif c in numeric_cols and stats.get(f"{c}__zero", 0) / row_count > zero_threshold:
            cols_to_drop.append(c)
        elif distinct_count <= 1:
            cols_to_drop.append(c)

    if cols_to_drop:
        print(f"Colonnes supprimées ({len(cols_to_drop)}) :")
        for c in cols_to_drop:
            print(f"  - {c} | missing={round(stats[f'{c}__missing']/row_count*100,1)}% | distinct={stats[f'{c}__distinct']}")

    return df.drop(*cols_to_drop), cols_to_drop


def save_table(df, table_name, display_result=True):
    """
    Sauvegarde un DataFrame dans le schéma team6 avec overwrite.
    """
    full_name = f"{TEAM_SCHEMA}.{table_name}" if "." not in table_name else table_name
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)
    print(f"✓ {full_name} ({df.count()} lignes)")
    if display_result:
        display(df.limit(5))
    return df


print("Fonctions utilitaires chargées : clean_strings(), drop_useless_columns(), save_table()")

Fonctions utilitaires chargées : clean_strings(), drop_useless_columns(), save_table()


# Nettoyage des données de livraisons (DLV)

Cette étape consiste à nettoyer et standardiser les données de livraisons des carrières.

Objectifs :
- harmoniser les types de données,
- supprimer les valeurs incohérentes,
- traiter les valeurs manquantes,
- préparer les indicateurs logistiques et financiers.

Ces données serviront ensuite :
- au calcul des volumes,
- à l’analyse des flux logistiques,
- aux indicateurs de chiffre d’affaires,
- et aux visualisations Power BI.

In [0]:
# ============================================================
# NETTOYAGE DLV LIVRAISONS
# ============================================================

df_raw = spark.table("efrei_estp_2026.input_data.dlv_deliveries_f")

# 1. Nettoyage strings (utilise la liste EMPTY_VALUES complète)
df = clean_strings(df_raw)

# 2. Sécuriser les types
df = (
    df
    .withColumn("DAT_DATE", F.to_date("DAT_DATE"))
    .withColumn("DAT_INVOICE_DATE", F.to_date("DAT_INVOICE_DATE"))
    .withColumn("DLV_TIMESTAMP", F.to_timestamp("DLV_TIMESTAMP"))
    .withColumn("DLV_QUANTITY", F.col("DLV_QUANTITY").cast("double"))
    .withColumn("DLV_UNIT_PRICE", F.col("DLV_UNIT_PRICE").cast("double"))
    .withColumn("DLV_QUANTITY_INVOICED", F.col("DLV_QUANTITY_INVOICED").cast("double"))
    .withColumn("DLV_PRE_TAX_AMOUNT_TRANSACTION", F.col("DLV_PRE_TAX_AMOUNT_TRANSACTION").cast("double"))
    .withColumn("DLV_PRE_TAX_AMOUNT", F.col("DLV_PRE_TAX_AMOUNT").cast("double"))
    .withColumn("DLV_UNIT_PRICE_TRANSACTION", F.col("DLV_UNIT_PRICE_TRANSACTION").cast("double"))
)

# 3. Suppression des lignes inutilisables
df = (
    df
    .filter(F.col("DAT_DATE").isNotNull())
    .filter(F.col("JBU_IDENTIFIER").isNotNull())
    .filter(F.col("STC_UP_IDENTIFIER").isNotNull())
    .filter(F.col("DLV_NUMBER_BL").isNotNull())
    .filter(F.col("DLV_LINE_NUMBER_BL").isNotNull())
)

# 4. Déduplication sur clé métier
business_key = ["JBU_IDENTIFIER", "DLV_NUMBER_BL", "DLV_LINE_NUMBER_BL"]
df = df.dropDuplicates(business_key)

# 5. Colonnes temporelles
df = (
    df
    .withColumn("year", F.year("DAT_DATE"))
    .withColumn("month", F.month("DAT_DATE"))
    .withColumn("quarter", F.quarter("DAT_DATE"))
    .withColumn("year_month", F.date_format("DAT_DATE", "yyyy-MM"))
)

# 6. Classification métier
df = (
    df
    .withColumn(
        "line_type",
        F.when(F.upper(F.col("PRD_IDENTIFIER")).like("%PORT%"), F.lit("TRANSPORT_FEE"))
         .when(F.upper(F.col("PRD_IDENTIFIER")).like("%TGAP%"), F.lit("TAX"))
         .when(F.upper(F.col("PRD_IDENTIFIER")).like("%CONTRIB%"), F.lit("CONTRIBUTION"))
         .when(F.upper(F.col("PRD_IDENTIFIER")).like("%ENERG%"), F.lit("ENERGY_CONTRIBUTION"))
         .when(F.col("DLV_PRE_TAX_AMOUNT") == 0, F.lit("ZERO_AMOUNT"))
         .otherwise(F.lit("PRODUCT"))
    )
    .withColumn(
        "has_transport",
        F.when(F.col("DLV_TRANSPORT_TYPE").isin("RENDU", "DEPART+PORT", "DÉPART + PORT"), F.lit(1))
         .otherwise(F.lit(0))
    )
    .withColumn(
        "is_reentered",
        F.when(F.col("DLV_IS_RE_ENTERED") == True, F.lit(1)).otherwise(F.lit(0))
    )
)

# 7. Indicateurs de qualité (on garde NULL au lieu de remplacer par 0)
df_nettoye = (
    df
    .withColumn(
        "is_valid_quantity",
        F.when(F.col("DLV_QUANTITY") > 0, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "is_valid_revenue",
        F.when(F.col("DLV_PRE_TAX_AMOUNT") > 0, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "computed_amount",
        F.col("DLV_QUANTITY_INVOICED") * F.col("DLV_UNIT_PRICE")
    )
    .withColumn(
        "amount_gap",
        F.abs(F.col("DLV_PRE_TAX_AMOUNT") - F.col("computed_amount"))
    )
)

# 8. Sauvegarde table détaillée
save_table(df_nettoye, "dlv_deliveries_f_clean")

# 9. KPI mensuel
df_dlv_kpi_monthly = (
    df_nettoye
    .groupBy(
        "STC_UP_IDENTIFIER", "JBU_IDENTIFIER", "year", "month",
        "year_month", "PRD_IDENTIFIER", "DLV_TRANSPORT_TYPE", "line_type", "CUR_NAME"
    )
    .agg(
        F.count("*").alias("nb_delivery_lines"),
        F.countDistinct("DLV_NUMBER_BL").alias("nb_delivery_notes"),
        F.countDistinct("TDP_CLIENT_MDM_IDENTIFIER").alias("nb_clients"),
        F.countDistinct("ITJ_IDENTIFIER").alias("nb_delivery_sites"),
        F.countDistinct("IAD_IDENTIFIER").alias("nb_delivery_addresses"),
        F.sum("DLV_QUANTITY").alias("total_quantity"),
        F.sum("DLV_QUANTITY_INVOICED").alias("total_quantity_invoiced"),
        F.sum("DLV_PRE_TAX_AMOUNT").alias("revenue_ht"),
        F.avg("DLV_UNIT_PRICE").alias("avg_unit_price"),
        F.sum("has_transport").alias("nb_transport_lines"),
        F.sum("is_reentered").alias("nb_reentered_lines"),
        F.avg("amount_gap").alias("avg_amount_gap")
    )
    .withColumn("revenue_per_quantity", F.when(F.col("total_quantity") > 0, F.col("revenue_ht") / F.col("total_quantity")))
    .withColumn("transport_line_rate", F.when(F.col("nb_delivery_lines") > 0, F.col("nb_transport_lines") / F.col("nb_delivery_lines")))
    .withColumn("reentered_line_rate", F.when(F.col("nb_delivery_lines") > 0, F.col("nb_reentered_lines") / F.col("nb_delivery_lines")))
)

save_table(df_dlv_kpi_monthly, "dlv_deliveries_kpi_monthly", display_result=False)

# 10. KPI annuel par site
df_dlv_site_yearly = (
    df_nettoye
    .groupBy("STC_UP_IDENTIFIER", "JBU_IDENTIFIER", "year", "CUR_NAME")
    .agg(
        F.count("*").alias("nb_delivery_lines"),
        F.countDistinct("DLV_NUMBER_BL").alias("nb_delivery_notes"),
        F.countDistinct("TDP_CLIENT_MDM_IDENTIFIER").alias("nb_clients"),
        F.countDistinct("PRD_IDENTIFIER").alias("nb_products"),
        F.sum("DLV_QUANTITY").alias("total_quantity"),
        F.sum("DLV_QUANTITY_INVOICED").alias("total_quantity_invoiced"),
        F.sum("DLV_PRE_TAX_AMOUNT").alias("revenue_ht"),
        F.sum("has_transport").alias("nb_transport_lines"),
        F.sum("is_reentered").alias("nb_reentered_lines")
    )
    .withColumn("avg_selling_price", F.when(F.col("total_quantity") > 0, F.col("revenue_ht") / F.col("total_quantity")))
    .withColumn("transport_line_rate", F.when(F.col("nb_delivery_lines") > 0, F.col("nb_transport_lines") / F.col("nb_delivery_lines")))
    .withColumn("reentered_line_rate", F.when(F.col("nb_delivery_lines") > 0, F.col("nb_reentered_lines") / F.col("nb_delivery_lines")))
)

save_table(df_dlv_site_yearly, "dlv_site_yearly_kpi", display_result=False)

# 11. KPI produits par année
df_dlv_product_yearly = (
    df_nettoye
    .filter(F.col("line_type") == "PRODUCT")
    .groupBy("STC_UP_IDENTIFIER", "year", "PRD_KEY", "PRD_IDENTIFIER", "CUR_NAME")
    .agg(
        F.sum("DLV_QUANTITY").alias("product_quantity"),
        F.sum("DLV_QUANTITY_INVOICED").alias("product_quantity_invoiced"),
        F.sum("DLV_PRE_TAX_AMOUNT").alias("product_revenue_ht"),
        F.countDistinct("TDP_CLIENT_MDM_IDENTIFIER").alias("nb_clients"),
        F.countDistinct("DLV_NUMBER_BL").alias("nb_delivery_notes")
    )
    .withColumn("avg_product_price", F.when(F.col("product_quantity") > 0, F.col("product_revenue_ht") / F.col("product_quantity")))
)

save_table(df_dlv_product_yearly, "dlv_product_yearly_kpi", display_result=False)

print("\n Pipeline DLV terminé : 4 tables créées")

✓ efrei_estp_2026.team6.dlv_deliveries_f_clean (77129802 lignes)


DLV_NUMBER_BL_KEY,JBU_IDENTIFIER,STC_UP_IDENTIFIER,DLV_NUMBER_BL,DLV_LINE_NUMBER_BL,DAT_DATE,DLV_TIMESTAMP,TDP_CLIENT_MDM_IDENTIFIER,PRD_KEY,PRD_IDENTIFIER,DLV_TRANSPORT_TYPE,ITJ_IDENTIFIER,DLV_UNIT_INVOICED,DLV_QUANTITY,DLV_UNIT_PRICE,DLV_QUANTITY_INVOICED,DLV_PRE_TAX_AMOUNT_TRANSACTION,DLV_PRE_TAX_AMOUNT,DLV_UNIT_PRICE_TRANSACTION,CUR_NAME,IAD_IDENTIFIER,DLV_ZEPHYR_INVOICE_NUMBER,COD_INVOICE_IDENTIFIER,DAT_INVOICE_DATE,DLV_IS_RE_ENTERED,year,month,quarter,year_month,line_type,has_transport,is_reentered,is_valid_quantity,is_valid_revenue,computed_amount,amount_gap
-8666366583227972042,2107601QQ001,UP1860,LC20020329C,90c9ca0a-9f30-4652-a735-b8baa0ee8039,2020-02-03,2020-02-03T07:20:50.000Z,341015,1581302102983613724,PORT_AG,DÉPART,CH14156C,TO,0.0,0.0,0.0,0.0,0.0,0.0,EUR,Cht.CH14156C.226149,null,null,null,null,2020,2,1,2020-02,TRANSPORT_FEE,0,0,0,0,0.0,0.0
7892886071277588763,1520005QQ008,UP283,LC18023970C,7d9240c4-2e0e-453f-b0cc-0b16079292f9,2018-02-28,2018-02-28T10:14:48.000Z,63470,3175067356223340202,TGAP,DÉPART,CH14673,TO,9.77,0.21,9.77,2.05,2.05,0.2,null,Cht.CH14673.124126,FC18000631P,FC18000631P,2018-02-28,false,2018,2,1,2018-02,TAX,0,0,1,1,2.0517,0.0017000000000000348
-1740687766557695566,2107601QQ001,UP1860,LC21057824C,90295e62-05a8-45ec-afe3-0c750fd10492,2021-05-20,2021-05-20T11:02:44.000Z,null,-2416333225787371343,TGAP,DÉPART,null,TO,6.37,0.0,6.37,0.0,0.0,0.0,EUR,Cli.BONCADEAUETPCC.239335,null,null,null,null,2021,5,2,2021-05,TAX,0,0,1,0,0.0,0.0
-1438482275256164670,1520005QQ002,UP276,LC19074658C,8a35e876-f17d-48d9-90a1-f3982d36ab79,2019-07-23,2019-07-23T16:41:46.000Z,88195,2117262319418067993,6/10BET,RENDU,CH16497C,TO,25.96,6.48,25.96,168.22,168.22,7.0,null,Cht.CH16497C.146275,FC19004113P,15200RX19023335,2019-07-31,false,2019,7,3,2019-07,PRODUCT,1,0,1,1,168.22080000000003,8.000000000265572E-4
2424416864240916998,1110902QQ004,UP427,LC21094660,a6ca9010-1b4f-47cd-b3d8-bccb74e1c317,2021-09-22,2021-09-22T10:01:05.000Z,40001,2617896122543072334,TGAP,DÉPART + PORT,CH13588,TO,30.83,0.21,30.83,6.47,6.47,0.21,EUR,Cht.CH13588.123866,FC21001918C,11109RX21024534,2021-09-30,false,2021,9,3,2021-09,TAX,1,0,1,1,6.4742999999999995,0.0042999999999997485


✓ efrei_estp_2026.team6.dlv_deliveries_kpi_monthly (1752364 lignes)
✓ efrei_estp_2026.team6.dlv_site_yearly_kpi (12203 lignes)
✓ efrei_estp_2026.team6.dlv_product_yearly_kpi (202816 lignes)

✅ Pipeline DLV terminé : 4 tables créées


# Nettoyage des données d’installations et de pannes (QIB)

Cette étape vise à analyser la santé opérationnelle des installations des carrières.

Les données de pannes permettent :
- d’identifier les équipements les plus critiques,
- de mesurer la disponibilité des installations,
- d’analyser la fréquence des incidents,
- et d’étudier les corrélations avec les risques climatiques.

Ces informations sont essentielles pour :
- la maintenance prédictive,
- l’optimisation des opérations,
- et le pilotage des performances industrielles.

In [0]:
# ============================================================
# NETTOYAGE QIB INSTALLATIONS / PANNES
# ============================================================

df_qib = spark.table("efrei_estp_2026.input_data.qib_quarries_installations_breakdowns_f")

print(f"Lignes initiales : {df_qib.count()}")
print(f"Colonnes initiales : {len(df_qib.columns)}")

# 1. Nettoyage strings
df_qib = clean_strings(df_qib)

# 2. Suppression colonnes inutiles (>80% NULL, constantes)
protected_qib = ["STC_UP_IDENTIFIER", "QIB_IDENTIFIER", "DAT_DATE", "QIB_START_HOUR", "QIB_END_HOUR"]
df_qib, dropped = drop_useless_columns(df_qib, protected_cols=protected_qib)

# 3. Déduplication sur clé métier (installation + date + heure début)
business_key_qib = [c for c in ["STC_UP_IDENTIFIER", "QIB_IDENTIFIER", "DAT_DATE", "QIB_START_HOUR"] if c in df_qib.columns]
if business_key_qib:
    nb_avant = df_qib.count()
    df_qib = df_qib.dropDuplicates(business_key_qib)
    print(f"Doublons supprimés : {nb_avant - df_qib.count()}")

# 4. Sauvegarde
save_table(df_qib, "qib_quarries_installations_breakdowns_clean")

Colonnes supprimées car plus de 80% de NULL / NaN :
Nombre de lignes supprimées après suppression des doublons : 0


QIB_IDENTIFIER,STC_UP_IDENTIFIER,DAT_DATE,QIB_START_HOUR,QIB_END_HOUR,QIB_TIME_BREAKDOWN
5245,UP1219,2026-02-13,13.00,16.00,3.00
5371,UP3214,2026-02-13,8.00,15.00,7.00
5376,UP561,2026-02-13,8.00,16.00,8.00
3520,UP4058,2025-10-21,7.00,7.58,0.58
5317,UP4048,2026-02-13,0.00,0.08,0.08
5273,UP5195,2026-02-13,7.00,8.00,1.00
3304,UP1021,2025-10-21,16.00,16.50,0.50
3301,UP1021,2025-10-21,11.50,12.00,0.50
3562,UP1353,2025-10-21,9.00,10.00,1.00
3462,UP1298,2025-10-21,0.00,8.00,8.00


# Nettoyage des données produits (PRD)

Les données produits permettent de caractériser les matériaux extraits et commercialisés par les carrières.

Cette étape permet :
- d’uniformiser les références produits,
- de structurer les catégories de granulats,
- et de préparer les analyses commerciales et logistiques.

Ces données seront utilisées dans :
- les analyses de volumes,
- les indicateurs de production,
- et les opportunités business liées aux flux matériaux.

In [0]:
# ============================================================
# NETTOYAGE PRD PRODUITS
# ============================================================

df = spark.table("efrei_estp_2026.input_data.prd_products_d")

print(f"Lignes initiales : {df.count()}")
print(f"Colonnes initiales : {len(df.columns)}")

# 1. Supprimer colonne inutile
if "JBU_IDENTIFIER" in df.columns:
    df = df.drop("JBU_IDENTIFIER")

# 2. Nettoyage strings
df = clean_strings(df)

# 3. Suppression colonnes inutiles
protected_prd = ["PRD_KEY", "PRD_IDENTIFIER", "PRD_LABEL", "PRD_UNIT", "PRD_TYPE", "PRD_DIRECTION"]
df, dropped = drop_useless_columns(df, protected_cols=protected_prd)

# 4. Filtrer lignes sans clé produit
if "PRD_KEY" in df.columns:
    df = df.filter(F.col("PRD_KEY").isNotNull())
if "PRD_IDENTIFIER" in df.columns:
    df = df.filter(F.col("PRD_IDENTIFIER").isNotNull())

# 5. Déduplication
if "PRD_KEY" in df.columns:
    df_clean = df.dropDuplicates(["PRD_KEY"])
else:
    df_clean = df.dropDuplicates(["PRD_IDENTIFIER"])

# 6. Normaliser le label produit
if "PRD_LABEL" in df_clean.columns:
    df_clean = (
        df_clean
        .withColumn("PRD_LABEL_NORMALIZED", F.lower(F.col("PRD_LABEL")))
        .withColumn("PRD_LABEL_NORMALIZED", F.regexp_replace("PRD_LABEL_NORMALIZED", "[^a-zA-Z0-9 ]", " "))
        .withColumn("PRD_LABEL_NORMALIZED", F.regexp_replace("PRD_LABEL_NORMALIZED", "\\s+", " "))
        .withColumn("PRD_LABEL_NORMALIZED", F.trim(F.col("PRD_LABEL_NORMALIZED")))
    )

# 7. Catégorie produit
if "PRD_LABEL_NORMALIZED" in df_clean.columns:
    df_clean = df_clean.withColumn(
        "product_family",
        F.when(F.col("PRD_LABEL_NORMALIZED").rlike("port|transport"), F.lit("TRANSPORT"))
         .when(F.col("PRD_LABEL_NORMALIZED").rlike("tgap|taxe"), F.lit("TAX"))
         .when(F.col("PRD_LABEL_NORMALIZED").rlike("energie|energ"), F.lit("ENERGY"))
         .when(F.col("PRD_LABEL_NORMALIZED").rlike("sable|sand"), F.lit("SAND"))
         .when(F.col("PRD_LABEL_NORMALIZED").rlike("grave|gravier|gravillon|gravel"), F.lit("GRAVEL"))
         .when(F.col("PRD_LABEL_NORMALIZED").rlike("beton|béton|concrete"), F.lit("CONCRETE"))
         .when(F.col("PRD_LABEL_NORMALIZED").rlike("enrobe|enrobé|asphalt"), F.lit("ASPHALT"))
         .otherwise(F.lit("OTHER"))
    )

# 8. Sauvegarde
save_table(df_clean, "prd_products_dim_clean")

Lignes initiales : 90514
Colonnes initiales : 7
Colonnes supprimées :
Table sauvegardée : efrei_estp_2026.team6.prd_products_dim_clean
Lignes finales : 90512
Colonnes finales : 8


PRD_IDENTIFIER,PRD_KEY,PRD_LABEL,PRD_TYPE,PRD_UNIT,PRD_DIRECTION,PRD_LABEL_NORMALIZED,product_family
0/3[BLANK],-5131354590787500461,Sable concassé calcaire 0/3,Produits,null,Sortant,sable concass calcaire 0 3,SAND
0/31.5,4146407884286209783,"Tout Venant 0/31,5",Produits,T,Sortant,tout venant 0 31 5,OTHER
57-2000010,505515316042046836,0/90Bundsikring BL II,Produits,T,Sortant,0 90bundsikring bl ii,OTHER
750006,-3224538149117518315,COLLINE YELLOW 100MM X 5M ROLL,Produits,U,Sortant,colline yellow 100mm x 5m roll,OTHER
CES20/40RSAV,-8472596681343702899,20/40 ROULE SILICO CALCAIRE,Produits,null,Sortant,20 40 roule silico calcaire,OTHER
EXT-PIQUET1,-8255423013582696157,PIQUET SCHISTE 1M SCIE 6-8 1BOUT SCIE,Produits,U,Sortant,piquet schiste 1m scie 6 8 1bout scie,OTHER
GSEICHE,6014592731563456544,GRAVIER SEICHE (T),Produits,null,Sortant,gravier seiche t,GRAVEL
S0/20SCBB,7896856292935851765,GRAVE 0/20 SILICO CALCAIRE EN BIG BAG,Produits,U,Sortant,grave 0 20 silico calcaire en big bag,GRAVEL
SCPLATRE,5938754560129616996,SC PLATRE,Produits,null,Sortant,sc platre,OTHER
SC_SABLE_SAJ,2999170301838563233,SABLE A JOINTS EN SAC 25KG,Produits,null,Sortant,sable a joints en sac 25kg,SAND


# Nettoyage des données sites et dimensions (STC)

Cette table constitue la dimension principale des carrières.

Elle centralise :
- les informations géographiques,
- les identifiants des sites,
- les structures organisationnelles,
- et les métadonnées des carrières.

Cette dimension permettra :
- la création des cartes Power BI,
- les analyses régionales,
- les croisements avec les données climatiques,
- et les analyses stratégiques multi-sites.

In [0]:
# ============================================================
# NETTOYAGE STC SITES (DIMENSION)
# ============================================================

df = spark.table("efrei_estp_2026.input_data.stc_colas_quarries_structures_d")

print(f"Lignes initiales : {df.count()}")
print(f"Colonnes initiales : {len(df.columns)}")

# 1. Nettoyage strings
df = clean_strings(df)

# 2. Filtrer lignes sans identifiant site
df = df.filter(F.col("STC_UP_IDENTIFIER").isNotNull())

# 3. Caster les coordonnées GPS
gps_cols = [
    "STC_UP_GPS_LONGITUDE", "STC_UP_GPS_LATITUDE",
    "STC_UE_GPS_LONGITUDE", "STC_UE_GPS_LATITUDE",
    "STC_ERT_GPS_LONGITUDE", "STC_ERT_GPS_LATITUDE"
]
for c in gps_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("double"))

# 4. Déduplication : garder la ligne la plus complète par site
useful_cols = [
    "STC_UP_LABEL", "STC_UP_ADDRESS_1", "STC_UP_ADDRESS_ZIPCODE",
    "STC_UP_ADDRESS_TOWN", "STC_UP_COUNTRY_ID", "STC_UP_COUNTRY_LABEL",
    "STC_UP_GPS_LONGITUDE", "STC_UP_GPS_LATITUDE",
    "STC_UP_ACTIVITY_1_LABEL_FR", "STC_UP_IS_ACTIVE",
    "STC_UE_IDENTIFIER", "STC_UE_LABEL", "STC_ERT_IDENTIFIER", "STC_ERT_LABEL"
]
existing_cols = [c for c in useful_cols if c in df.columns]

df = df.withColumn(
    "completeness_score",
    sum([F.when(F.col(c).isNotNull(), F.lit(1)).otherwise(F.lit(0)) for c in existing_cols])
)

window_site = Window.partitionBy("STC_UP_IDENTIFIER").orderBy(F.desc("completeness_score"))
df = (
    df.withColumn("rn", F.row_number().over(window_site))
    .filter(F.col("rn") == 1)
    .drop("rn", "completeness_score")
)

# 5. Sélectionner colonnes BI
cols_sites = [
    "STC_UP_IDENTIFIER", "STC_UP_LABEL",
    "STC_UP_ADDRESS_1", "STC_UP_ADDRESS_2", "STC_UP_ADDRESS_ZIPCODE",
    "STC_UP_ADDRESS_TOWN", "STC_UP_COUNTRY_ID", "STC_UP_COUNTRY_LABEL",
    "STC_UP_GPS_LONGITUDE", "STC_UP_GPS_LATITUDE",
    "STC_UP_ACTIVITY_1_CODE", "STC_UP_ACTIVITY_1_LABEL_FR", "STC_UP_ACTIVITY_1_LABEL_EN",
    "STC_UP_ACTIVITY_2_CODE", "STC_UP_ACTIVITY_2_LABEL_FR", "STC_UP_ACTIVITY_2_LABEL_EN",
    "STC_UP_ACTIVITY_3_CODE", "STC_UP_ACTIVITY_3_LABEL_FR", "STC_UP_ACTIVITY_3_LABEL_EN",
    "STC_UP_ACTIVITY_4_CODE", "STC_UP_ACTIVITY_4_LABEL_FR", "STC_UP_ACTIVITY_4_LABEL_EN",
    "STC_UP_IS_ACTIVE",
    "STC_UE_IDENTIFIER", "STC_UE_LABEL", "STC_UE_ADDRESS_TOWN",
    "STC_UE_COUNTRY_ID_ISO2", "STC_UE_GPS_LONGITUDE", "STC_UE_GPS_LATITUDE",
    "STC_ERT_IDENTIFIER", "STC_ERT_LABEL", "STC_ERT_ADDRESS_TOWN",
    "STC_ERT_COUNTRY_ID_ISO2", "STC_ERT_GPS_LONGITUDE", "STC_ERT_GPS_LATITUDE"
]
df_sites_dim = df.select(*[c for c in cols_sites if c in df.columns])

# 6. Colonnes BI dérivées
if "STC_UP_GPS_LONGITUDE" in df_sites_dim.columns:
    df_sites_dim = df_sites_dim.withColumn(
        "has_up_gps",
        F.when(
            F.col("STC_UP_GPS_LONGITUDE").isNotNull()
            & F.col("STC_UP_GPS_LATITUDE").isNotNull()
            & ~F.isnan(F.col("STC_UP_GPS_LONGITUDE"))
            & ~F.isnan(F.col("STC_UP_GPS_LATITUDE")),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

if "STC_UP_IS_ACTIVE" in df_sites_dim.columns:
    df_sites_dim = df_sites_dim.withColumn(
        "site_status",
        F.when(F.col("STC_UP_IS_ACTIVE") == True, F.lit("Active"))
         .when(F.col("STC_UP_IS_ACTIVE") == False, F.lit("Closed"))
         .otherwise(F.lit("Unknown"))
    )

if "STC_UP_LABEL" in df_sites_dim.columns:
    df_sites_dim = df_sites_dim.withColumn(
        "site_display_name",
        F.when(F.col("STC_UP_LABEL").isNotNull(), F.col("STC_UP_LABEL"))
         .otherwise(F.col("STC_UP_IDENTIFIER"))
    )

# 7. Sauvegarde
save_table(df_sites_dim, "stc_sites_dim_clean")

Nombre de lignes initial : 1030
Nombre de colonnes initial : 36
Nombre de lignes final : 1030
Nombre de colonnes final : 38
Colonnes finales :
- STC_UP_IDENTIFIER
- STC_UP_LABEL
- STC_UP_ADDRESS_1
- STC_UP_ADDRESS_2
- STC_UP_ADDRESS_ZIPCODE
- STC_UP_ADDRESS_TOWN
- STC_UP_COUNTRY_ID
- STC_UP_COUNTRY_LABEL
- STC_UP_GPS_LONGITUDE
- STC_UP_GPS_LATITUDE
- STC_UP_ACTIVITY_1_CODE
- STC_UP_ACTIVITY_1_LABEL_FR
- STC_UP_ACTIVITY_1_LABEL_EN
- STC_UP_ACTIVITY_2_CODE
- STC_UP_ACTIVITY_2_LABEL_FR
- STC_UP_ACTIVITY_2_LABEL_EN
- STC_UP_ACTIVITY_3_CODE
- STC_UP_ACTIVITY_3_LABEL_FR
- STC_UP_ACTIVITY_3_LABEL_EN
- STC_UP_ACTIVITY_4_CODE
- STC_UP_ACTIVITY_4_LABEL_FR
- STC_UP_ACTIVITY_4_LABEL_EN
- STC_UP_IS_ACTIVE
- STC_UE_IDENTIFIER
- STC_UE_LABEL
- STC_UE_ADDRESS_TOWN
- STC_UE_COUNTRY_ID_ISO2
- STC_UE_GPS_LONGITUDE
- STC_UE_GPS_LATITUDE
- STC_ERT_IDENTIFIER
- STC_ERT_LABEL
- STC_ERT_ADDRESS_TOWN
- STC_ERT_COUNTRY_ID_ISO2
- STC_ERT_GPS_LONGITUDE
- STC_ERT_GPS_LATITUDE
- has_up_gps
- site_status
- site_disp

STC_UP_IDENTIFIER,STC_UP_LABEL,STC_UP_ADDRESS_1,STC_UP_ADDRESS_2,STC_UP_ADDRESS_ZIPCODE,STC_UP_ADDRESS_TOWN,STC_UP_COUNTRY_ID,STC_UP_COUNTRY_LABEL,STC_UP_GPS_LONGITUDE,STC_UP_GPS_LATITUDE,STC_UP_ACTIVITY_1_CODE,STC_UP_ACTIVITY_1_LABEL_FR,STC_UP_ACTIVITY_1_LABEL_EN,STC_UP_ACTIVITY_2_CODE,STC_UP_ACTIVITY_2_LABEL_FR,STC_UP_ACTIVITY_2_LABEL_EN,STC_UP_ACTIVITY_3_CODE,STC_UP_ACTIVITY_3_LABEL_FR,STC_UP_ACTIVITY_3_LABEL_EN,STC_UP_ACTIVITY_4_CODE,STC_UP_ACTIVITY_4_LABEL_FR,STC_UP_ACTIVITY_4_LABEL_EN,STC_UP_IS_ACTIVE,STC_UE_IDENTIFIER,STC_UE_LABEL,STC_UE_ADDRESS_TOWN,STC_UE_COUNTRY_ID_ISO2,STC_UE_GPS_LONGITUDE,STC_UE_GPS_LATITUDE,STC_ERT_IDENTIFIER,STC_ERT_LABEL,STC_ERT_ADDRESS_TOWN,STC_ERT_COUNTRY_ID_ISO2,STC_ERT_GPS_LONGITUDE,STC_ERT_GPS_LATITUDE,has_up_gps,site_status,site_display_name
UP100,HLW YVOIR,GRAND ROUTE 71,null,B-4367,CRISNEE,BEL,Belgique,5.400450229644775,50.707908630371094,TE,Dépôts pour négoce,Depot for trading,null,null,null,null,null,null,null,null,null,true,UE1297,HLW,CRISNEE,BE,null,null,ERT22,BELGIUM,null,BE,null,null,1,Active,HLW YVOIR
UP1007,ARDECHE ENROBES LAVILLEDIEU,BOIS REDON,null,07170,LAVILLEDIEU,FRA,France,4.436369895935059,44.593719482421875,TE,Dépôts pour négoce,Depot for trading,null,null,null,null,null,null,null,null,null,true,UE603,ARDECHE ENROBES,LAVILLEDIEU,FR,4.500909805297852,44.55657958984375,ERT5,TERRITOIRE SUD EST,null,FR,null,null,1,Active,ARDECHE ENROBES LAVILLEDIEU
UP1010,CARRIERE DE LURCY LEVIS,LIEUDIT LES BAUDRANS,null,03320,LURCY LEVIS,FRA,France,2.909670114517212,46.71144104003906,QQ,"Carrières, gravières",Quarries,TE,Dépôts pour négoce,Depot for trading,null,null,null,null,null,null,true,UE610,NEXSTONE AUVERGNE NORD,BRANSAT,FR,3.2292399406433105,46.32102966308594,ERT5,TERRITOIRE SUD EST,null,FR,null,null,1,Active,CARRIERE DE LURCY LEVIS
UP1011,CARRIERE DE CERILLY,PERNIER,null,03350,CERILLY,FRA,France,2.7853200435638428,46.589778900146484,TE,Dépôts pour négoce,Depot for trading,null,null,null,null,null,null,null,null,null,true,UE610,NEXSTONE AUVERGNE NORD,BRANSAT,FR,3.2292399406433105,46.32102966308594,ERT5,TERRITOIRE SUD EST,null,FR,null,null,1,Active,CARRIERE DE CERILLY
UP1014,CARRIERE DE BUXIÈRES,LIEUDIT LES ROCS,null,03440,BUXIERES LES MINES,FRA,France,2.912450075149536,46.450199127197266,null,null,null,null,null,null,null,null,null,null,null,null,true,UE610,NEXSTONE AUVERGNE NORD,BRANSAT,FR,3.2292399406433105,46.32102966308594,ERT5,TERRITOIRE SUD EST,null,FR,null,null,1,Active,CARRIERE DE BUXIÈRES
UP1015,SITE D'EVAUX,LES DOULAUDS,null,23110,EVAUX LES BAINS,FRA,France,2.464859962463379,46.183990478515625,null,null,null,null,null,null,null,null,null,null,null,null,true,UE610,NEXSTONE AUVERGNE NORD,BRANSAT,FR,3.2292399406433105,46.32102966308594,ERT5,TERRITOIRE SUD EST,null,FR,null,null,1,Active,SITE D'EVAUX
UP1016,CARRIERE DE MEILLERS,BRUYERE CHAUMIAU,null,03210,MEILLERS,FRA,France,3.105530023574829,46.5224609375,TE,Dépôts pour négoce,Depot for trading,null,null,null,null,null,null,null,null,null,true,UE610,NEXSTONE AUVERGNE NORD,BRANSAT,FR,3.2292399406433105,46.32102966308594,ERT5,TERRITOIRE SUD EST,null,FR,null,null,1,Active,CARRIERE DE MEILLERS
UP1017,CARRIERE DE MONTAIGUT,LIEUDIT BROSSE,null,63700,MONTAIGUT EN COMBRAILLE,FRA,France,2.8046200275421143,46.176761627197266,TE,Dépôts pour négoce,Depot for trading,null,null,null,null,null,null,null,null,null,true,UE610,NEXSTONE AUVERGNE NORD,BRANSAT,FR,3.2292399406433105,46.32102966308594,ERT5,TERRITOIRE SUD EST,null,FR,null,null,1,Active,CARRIERE DE MONTAIGUT
UP1019,CARRIERE DE PERROGUIN BRANSAT,ROC PERROGUIN,null,03500,BRANSAT,FRA,France,3.2332499027252197,46.31412887573242,TE,Dépôts pour négoce,Depot for trading,null,null,null,null,null,null,null,null,null,true,UE610,NEXSTONE AUVERGNE NORD,BRANSAT,FR,3.2292399406433105,46.32102966308594,ERT5,TERRITOIRE SUD EST,null,FR,null,null,1,Active,CARRIERE DE PERROGUIN BRANSAT
UP1020,CARRIERE DE VENSAT,LA COTE ROUSSE,null,63260,VENSAT,FRA,France,3.179379940032959,46.048919677734375,TE,Dépôts pou

Table sauvegardée : efrei_estp_2026.team6.stc_sites_dim_clean


# Intégration des données climatiques EcoAct

Cette étape permet d’intégrer des données externes d’exposition aux risques climatiques.

Les risques étudiés incluent notamment :
- les incendies,
- les sécheresses,
- les fortes chaleurs,
- les inondations,
- et d’autres événements climatiques extrêmes.

L’objectif est de :
- croiser les risques climatiques avec les performances des carrières,
- identifier les sites les plus exposés,
- et alimenter les analyses de durabilité et de résilience.

Ces données jouent un rôle central dans les business opportunities liées :
- à la maintenance prédictive,
- à la gestion des risques,
- et à la durabilité des infrastructures.

In [0]:
# ============================================================
# INGESTION CSV ECOACT (RISQUES CLIMATIQUES)
# ============================================================

# 1. Lecture du CSV
df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .load("/Volumes/efrei_estp_2026/input_data/ecoact_data")
)

print(f"Fichier chargé : {df.count()} lignes, {len(df.columns)} colonnes")

# 2. Nettoyage des noms de colonnes
new_columns = []
for col_name in df.columns:
    clean_name = (
        col_name.strip()
        .replace(" ", "_").replace("-", "_").replace("/", "_")
        .replace("(", "").replace(")", "").replace(".", "_")
        .replace("%", "PERCENT")
        .upper()
    )
    new_columns.append(clean_name)

df = df.toDF(*new_columns)

# 3. Nettoyage strings (utilise la fonction utilitaire)
df = clean_strings(df)

# 4. Suppression des doublons complets et lignes vides
df = df.dropDuplicates()
df = df.na.drop(how="all")

# 5. Suppression colonnes inutiles (>80% NULL)
df, dropped = drop_useless_columns(df, protected_cols=[])

print(f"Résultat final : {df.count()} lignes, {len(df.columns)} colonnes")

# 6. Sauvegarde
save_table(df, "ecoact_data_clean")

Fichier chargé avec succès.
Nombre de lignes : 120858
Nombre de colonnes : 30
Colonnes renommées.
Nombre de lignes supprimées après suppression des doublons : 0
Nombre de lignes totalement vides supprimées : 0
Espaces supprimés dans les colonnes texte.
Valeurs vides remplacées par NULL.
Colonnes texte normalisées.
Colonne _C29_NORMALIZED supprimée.

DOUBLONS METIER
SITE_ID : 552 valeurs dupliquées
CLIENT_NAME : 1 valeurs dupliquées
LAT : 216 valeurs dupliquées
LON : 216 valeurs dupliquées
INDICATOR : 44 valeurs dupliquées

VALEURS NULL
SITE_ID : 0 valeurs nulles
CLIENT_NAME : 0 valeurs nulles
LAT : 0 valeurs nulles
LON : 0 valeurs nulles
INDICATOR : 0 valeurs nulles
UNIT : 0 valeurs nulles
PARAMETER : 0 valeurs nulles
PARAMETER_UNIT : 0 valeurs nulles
SCENARIO : 0 valeurs nulles
REFERENCE_PERIOD : 0 valeurs nulles
FUTURE_PERIOD : 0 valeurs nulles
REFERENCE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS : 0 valeurs nulles
FUTURE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS : 0 valeurs nulles
REFERENCE_PERIOD_Y

SITE_ID,CLIENT_NAME,LAT,LON,INDICATOR,UNIT,PARAMETER,PARAMETER_UNIT,SCENARIO,REFERENCE_PERIOD,FUTURE_PERIOD,REFERENCE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS,FUTURE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS,REFERENCE_PERIOD_YEAR_STD_MEDIAN_OF_MODELS,FUTURE_PERIOD_YEAR_STD_MEDIAN_OF_MODELS,NUMBER_OF_MODELS,MODEL_CONFIDENCE,ANOMALY_MEAN_FUTURE_MINUS_REF,ANOMALY_STD_FUTURE_MINUS_REF,REFERENCE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_RAW,REFERENCE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_LABEL,FUTURE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_RAW,FUTURE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_LABEL,SITE_NAME,SITE_TYPOLOGY,STRUCTURE_CODE,LIBELLÉ_STRUCTURE,STRUCTURE_ACTIVITE,EU_TAXONOMY_HAZARD,_C29,CLIENT_NAME_NORMALIZED,LAT_NORMALIZED,LON_NORMALIZED,INDICATOR_NORMALIZED,UNIT_NORMALIZED,PARAMETER_NORMALIZED,PARAMETER_UNIT_NORMALIZED,SCENARIO_NORMALIZED,REFERENCE_PERIOD_NORMALIZED,FUTURE_PERIOD_NORMALIZED,REFERENCE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS_NORMALIZED,FUTURE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS_NORMALIZED,REFERENCE_PERIOD_YEAR_STD_MEDIAN_OF_MODELS_NORMALIZED,FUTURE_PERIOD_YEAR_STD_MEDIAN_OF_MODELS_NORMALIZED,MODEL_CONFIDENCE_NORMALIZED,ANOMALY_MEAN_FUTURE_MINUS_REF_NORMALIZED,ANOMALY_STD_FUTURE_MINUS_REF_NORMALIZED,REFERENCE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_RAW_NORMALIZED,REFERENCE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_LABEL_NORMALIZED,FUTURE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_RAW_NORMALIZED,FUTURE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_LABEL_NORMALIZED,SITE_NAME_NORMALIZED,SITE_TYPOLOGY_NORMALIZED,STRUCTURE_CODE_NORMALIZED,LIBELLÉ_STRUCTURE_NORMALIZED,STRUCTURE_ACTIVITE_NORMALIZED,EU_TAXONOMY_HAZARD_NORMALIZED
38,Colas,"42,55707","9,49079",cold_solifluction_chronic_trend,days,0,°C,rcp85,1971-2000,2021-2040,21,"10,9","8,734605505","5,99666574",7,high,"-10,1","-2,737939765","52,17",medium,"43,39",medium,CICO,Carriere,UE2142,CNB,BPE,Solifluction,null,colas,42 55707,9 49079,cold solifluction chronic trend,days,0,c,rcp85,1971 2000,2021 2040,21,10 9,8 734605505,5 99666574,high,10 1,2 737939765,52 17,medium,43 39,medium,cico,carriere,ue2142,cnb,bpe,solifluction
39,Colas,"42,55707","9,49079",cold_solifluction_chronic_trend,days,0,°C,rcp85,1971-2000,2021-2040,21,"10,9","8,734605505","5,99666574",7,high,"-10,1","-2,737939765","52,17",medium,"43,39",medium,CICO,Carriere,UE2156,SEP CNB,BPE,Solifluction,null,colas,42 55707,9 49079,cold solifluction chronic trend,days,0,c,rcp85,1971 2000,2021 2040,21,10 9,8 734605505,5 99666574,high,10 1,2 737939765,52 17,medium,43 39,medium,cico,carriere,ue2156,sep cnb,bpe,solifluction
39,Colas,"42,55707","9,49079",wind_low_wind_acute_frequency,days,3,m/s,rcp26,1971-2000,2081-2100,"265,8333333","266,9473684","11,70269418","11,44619009",6,low,"1,114035088","-0,256504088","89,54",high,"89,66",high,CICO,Carriere,UE2156,SEP CNB,BPE,n.a.,null,colas,42 55707,9 49079,wind low wind acute frequency,days,3,m s,rcp26,1971 2000,2081 2100,265 8333333,266 9473684,11 70269418,11 44619009,low,1 114035088,0 256504088,89 54,high,89 66,high,cico,carriere,ue2156,sep cnb,bpe,n a
39,Colas,"42,55707","9,49079",heat_heat_wave_threshold,days,35,degree,rcp85,1971-2000,2081-2100,"1,283333333","16,13157895","2,346066084","8,449322203",6,high,"14,84824561","6,103256118","8,56",low,"47,94",medium,CICO,Carriere,UE2156,SEP CNB,BPE,Heat wave,null,colas,42 55707,9 49079,heat heat wave threshold,days,35,degree,rcp85,1971 2000,2081 2100,1 283333333,16 13157895,2 346066084,8 449322203,high,14 84824561,6 103256118,8 56,low,47 94,medium,cico,carriere,ue2156,sep cnb,bpe,heat wave
40,Colas,"42,55707","9,49079",wind_tornadoes_acute_frequency,no unit,150,km,historical,not applicable,not applicable,77,-999999,-999999,-999999,1,low,-999999,-999999,70,medium - high,70,medium - high,CICO,Carriere,UP3257,CNBSP- SITE DE BORGO,Blocs de Beton,Tornado,null,colas,42 55707,9 49079,wind tornadoes acute frequency,no unit,150,km,historical,not applicable,not applicable,77,999999,999999,999999,low,999999,999999,70,medium high,70,medium high,cico,carriere,up3257,cnbsp site de 

# Construction des indicateurs de production annuelle (QYP)

Cette étape permet de calculer les KPI de production des carrières.

Les indicateurs produits incluent notamment :
- les volumes annuels,
- les tendances de production,
- les évolutions temporelles,
- et les comparaisons entre carrières.

Ces KPI permettront :
- d’alimenter les dashboards stratégiques,
- d’identifier les sites les plus performants,
- et d’aider à la prise de décision opérationnelle.

In [0]:
# ============================================================
# NETTOYAGE QYP PRODUCTION ANNUELLE + KPIs
# ============================================================

df = spark.table("efrei_estp_2026.input_data.qyp_quarries_yearly_reporting_f")

# 1. Nettoyage strings
df = clean_strings(df)

# 2. Types + clés métier
df = df.withColumn("DAT_YEAR", F.col("DAT_YEAR").cast("int"))
df = df.filter(F.col("STC_UP_IDENTIFIER").isNotNull()).filter(F.col("DAT_YEAR").isNotNull())
df = df.dropDuplicates(["STC_UP_IDENTIFIER", "DAT_YEAR"])

# 3. Suppression colonnes inutiles
protected_qyp = ["STC_UP_IDENTIFIER", "DAT_YEAR", "CUR_NAME", "CUR_REPORTING_NAME"]
df_clean, dropped = drop_useless_columns(df, protected_cols=protected_qyp)

# 4. KPIs QYP
df_qyp_kpi = df_clean

# Volume de production principal
if "QYP_PRODUCTION_VOLUME_METRICS" in df_qyp_kpi.columns:
    df_qyp_kpi = df_qyp_kpi.withColumn("production_volume", F.col("QYP_PRODUCTION_VOLUME_METRICS"))
elif "QYP_PRODUCTION_VOLUME_LOCAL" in df_qyp_kpi.columns:
    df_qyp_kpi = df_qyp_kpi.withColumn("production_volume", F.col("QYP_PRODUCTION_VOLUME_LOCAL"))
else:
    df_qyp_kpi = df_qyp_kpi.withColumn("production_volume", F.lit(None).cast("double"))

# Coût d'extraction par unité
if "QYP_EXTRACTION_ANNUAL_FEES_REPORTING" in df_qyp_kpi.columns:
    df_qyp_kpi = df_qyp_kpi.withColumn(
        "extraction_cost_per_unit",
        F.when(F.col("production_volume") > 0, F.col("QYP_EXTRACTION_ANNUAL_FEES_REPORTING") / F.col("production_volume"))
    )

# Valeur nette gisement par unité
if "QYP_NET_BOOK_VALUE_FOR_DEPOSIT_LAND_REPORTING" in df_qyp_kpi.columns:
    df_qyp_kpi = df_qyp_kpi.withColumn(
        "deposit_net_book_value_per_unit",
        F.when(F.col("production_volume") > 0, F.col("QYP_NET_BOOK_VALUE_FOR_DEPOSIT_LAND_REPORTING") / F.col("production_volume"))
    )

# Taux de dépréciation
if "QYP_CURRENT_YEAR_LAND_DEPOSIT_DEPRECIATION_REPORTING" in df_qyp_kpi.columns and "QYP_LAND_DEPOSIT_ACQUISITION_VALUE_REPORTING" in df_qyp_kpi.columns:
    df_qyp_kpi = df_qyp_kpi.withColumn(
        "depreciation_rate",
        F.when(F.col("QYP_LAND_DEPOSIT_ACQUISITION_VALUE_REPORTING") > 0,
               F.col("QYP_CURRENT_YEAR_LAND_DEPOSIT_DEPRECIATION_REPORTING") / F.col("QYP_LAND_DEPOSIT_ACQUISITION_VALUE_REPORTING"))
    )

# Taux de valeur comptable restante
if "QYP_NET_BOOK_VALUE_FOR_DEPOSIT_LAND_REPORTING" in df_qyp_kpi.columns and "QYP_LAND_DEPOSIT_ACQUISITION_VALUE_REPORTING" in df_qyp_kpi.columns:
    df_qyp_kpi = df_qyp_kpi.withColumn(
        "remaining_book_value_rate",
        F.when(F.col("QYP_LAND_DEPOSIT_ACQUISITION_VALUE_REPORTING") > 0,
               F.col("QYP_NET_BOOK_VALUE_FOR_DEPOSIT_LAND_REPORTING") / F.col("QYP_LAND_DEPOSIT_ACQUISITION_VALUE_REPORTING"))
    )

df_qyp_kpi = df_qyp_kpi.withColumn(
    "is_active_production",
    F.when(F.col("production_volume") > 0, F.lit(1)).otherwise(F.lit(0))
)

# Sélection colonnes BI
cols_qyp_bi = [
    "STC_UP_IDENTIFIER", "DAT_YEAR", "CUR_NAME", "CUR_REPORTING_NAME",
    "production_volume", "QYP_PRODUCTION_VOLUME_CONSOLIDATED",
    "QYP_EXTRACTION_ANNUAL_FEES_REPORTING", "QYP_NET_BOOK_VALUE_FOR_DEPOSIT_LAND_REPORTING",
    "QYP_CURRENT_YEAR_LAND_DEPOSIT_DEPRECIATION_REPORTING", "QYP_LAND_DEPOSIT_ACQUISITION_VALUE_REPORTING",
    "extraction_cost_per_unit", "deposit_net_book_value_per_unit",
    "depreciation_rate", "remaining_book_value_rate", "is_active_production"
]
df_qyp_kpi = df_qyp_kpi.select(*[c for c in cols_qyp_bi if c in df_qyp_kpi.columns])

# 5. Sauvegarde
save_table(df_clean, f"{TEAM_SCHEMA}.qyp_quarries_yearly_reporting_clean", display_result=False)
save_table(df_qyp_kpi, f"{TEAM_SCHEMA}.qyp_quarries_yearly_kpi")

QYP colonnes supprimées :
['QYP_LAND_DEPOSIT_ACQUISITION_VALUE_LOCAL', 'QYP_LAND_DEPOSIT_ACQUISITION_VALUE_REPORTING', 'QYP_LAND_DEPOSIT_ACQUISITION_VALUE_CONSOLIDATED']
Tables QYP créées :
- efrei_estp_2026.team6.qyp_quarries_yearly_reporting_clean
- efrei_estp_2026.team6.qyp_quarries_yearly_kpi


STC_UP_IDENTIFIER,DAT_YEAR,CUR_NAME,CUR_REPORTING_NAME,production_volume,QYP_PRODUCTION_VOLUME_CONSOLIDATED,QYP_EXTRACTION_ANNUAL_FEES_REPORTING,QYP_NET_BOOK_VALUE_FOR_DEPOSIT_LAND_REPORTING,QYP_CURRENT_YEAR_LAND_DEPOSIT_DEPRECIATION_REPORTING,extraction_cost_per_unit,deposit_net_book_value_per_unit,is_active_production
UP1151,2023,EUR,EUR,149287.00,149287.00,0.00,0.00,0.00,0E-13,0E-13,1
UP1524,2023,EUR,EUR,24536.00,24536.00,0.00,0.00,0.00,0E-13,0E-13,1
UP4014,2022,EUR,EUR,57105.00,57105.00,0.00,0.00,0.00,0E-13,0E-13,1
UP1709,2025,MUR,EUR,900233.10,449576.41,856031.98,0.00,0.00,0.9509003612509,0E-13,1
UP3630,2025,CAD,EUR,91266.17,91266.17,0.00,0.00,0.00,0E-13,0E-13,1
UP1427,2023,CZK,EUR,47608.00,47174.77,0.00,0.00,0.00,0E-13,0E-13,1
UP2316,2024,CAD,EUR,51138.84,51138.84,0.00,0.00,0.00,0E-13,0E-13,1
UP2375,2023,CAD,EUR,3006.62,3006.62,0.00,0.00,0.00,0E-13,0E-13,1
UP2748,2023,USD,EUR,115753.61,115753.61,0.00,0.00,0.00,0E-13,0E-13,1
UP3534,2025,CAD,EUR,0.00,0.00,0.00,0.00,0.00,null,null,0


# Construction des indicateurs de réserves et durabilité (RYP)

Cette étape permet d’analyser les réserves disponibles des carrières ainsi que leur durabilité dans le temps.

Les indicateurs calculés permettent :
- d’estimer la durée de vie des carrières,
- de suivre les taux de stérile,
- d’analyser les capacités restantes,
- et d’anticiper les besoins futurs.

Ces données sont particulièrement importantes dans une logique :
- de planification stratégique,
- de gestion durable des ressources,
- et d’anticipation des risques liés aux scénarios climatiques.

In [0]:
# ============================================================
# NETTOYAGE RYP RÉSERVES ANNUELLES + KPIs
# ============================================================

df = spark.table("efrei_estp_2026.input_data.ryp_quarries_reserves_yearly_reporting_f")

print(f"Lignes initiales : {df.count()}, Colonnes : {len(df.columns)}")

# 1. Nettoyage strings
df = clean_strings(df)

# 2. Types + clés métier
if "DAT_YEAR" in df.columns:
    df = df.withColumn("DAT_YEAR", F.col("DAT_YEAR").cast("int"))

df = df.filter(F.col("STC_UP_IDENTIFIER").isNotNull()).filter(F.col("DAT_YEAR").isNotNull())
df = df.dropDuplicates(["STC_UP_IDENTIFIER", "DAT_YEAR"])

# 3. Suppression colonnes inutiles
protected_ryp = ["STC_UP_IDENTIFIER", "DAT_YEAR", "CUR_NAME", "CUR_REPORTING_NAME"]
df_clean, dropped = drop_useless_columns(df, protected_cols=protected_ryp)

# 4. KPIs RYP - volume de réserve
numeric_cols_clean = [f.name for f in df_clean.schema.fields if isinstance(f.dataType, NumericType)]

reserve_candidate_cols = [
    c for c in numeric_cols_clean
    if any(kw in c.upper() for kw in ["RESERVE", "RESERV", "VOLUME", "TON", "QTY", "QUANTITY"])
]

df_ryp_kpi = df_clean

if reserve_candidate_cols:
    df_ryp_kpi = df_ryp_kpi.withColumn(
        "reserve_volume",
        F.coalesce(*[F.col(c).cast("double") for c in reserve_candidate_cols])
    )
else:
    df_ryp_kpi = df_ryp_kpi.withColumn("reserve_volume", F.lit(None).cast("double"))

df_ryp_kpi = df_ryp_kpi.withColumn(
    "has_reserve_data",
    F.when(F.col("reserve_volume").isNotNull(), F.lit(1)).otherwise(F.lit(0))
)

# Sélection colonnes BI
cols_ryp_bi = ["STC_UP_IDENTIFIER", "DAT_YEAR", "CUR_NAME", "CUR_REPORTING_NAME", "reserve_volume", "has_reserve_data"] + reserve_candidate_cols
cols_ryp_bi_unique = list(dict.fromkeys(c for c in cols_ryp_bi if c in df_ryp_kpi.columns))
df_ryp_kpi = df_ryp_kpi.select(*cols_ryp_bi_unique)

# 5. Sauvegarde
save_table(df_clean, f"{TEAM_SCHEMA}.ryp_quarries_reserves_yearly_reporting_clean", display_result=False)
save_table(df_ryp_kpi, f"{TEAM_SCHEMA}.ryp_quarries_reserves_yearly_kpi")

Nombre de lignes initial : 2659
Nombre de colonnes initial : 203
Diagnostic colonnes :
KEEP - RES_IDENTIFIER | null_nan=0.0% | top_value=41916 | top_rate=0.13% | distinct=868
KEEP - DAT_YEAR | null_nan=0.0% | top_value=2023 | top_rate=34.72% | distinct=3
KEEP - STC_UP_IDENTIFIER | null_nan=0.0% | top_value=UP901 | top_rate=0.13% | distinct=833
DROP - RYP_REPORTING_METHOD_CODE | valeur dominante >95% : OLD (96.95%)
DROP - USR_COMPETENT_PERSON_LOGIN | >80% NULL/NaN (99.75%)
KEEP - RYP_REMAINING_OVERBURDEN_VOLUME_LOCAL | null_nan=0.0% | top_value=0.00 | top_rate=92.58% | distinct=95
KEEP - RYP_REMAINING_OVERBURDEN_SURFACE_LOCAL | null_nan=0.0% | top_value=0.00 | top_rate=92.03% | distinct=104
KEEP - RYP_WASTE_PERCENTAGE | null_nan=0.0% | top_value=0.00 | top_rate=93.13% | distinct=23
KEEP - DAT_LICENSE_END_DATE | null_nan=45.23% | top_value=None | top_rate=45.23% | distinct=502
KEEP - RYP_ANNUAL_TONNAGE_LICENSE_MAX_LOCAL | null_nan=0.0% | top_value=0.00 | top_rate=52.61% | distinct=169
KE

STC_UP_IDENTIFIER,DAT_YEAR,CUR_NAME,reserve_volume,has_reserve_data,RYP_REMAINING_OVERBURDEN_VOLUME_LOCAL,RYP_ANNUAL_TONNAGE_LICENSE_MAX_LOCAL,RYP_ANNUAL_TONNAGE_LICENSE_AVERAGE_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_LOCAL,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_LOCAL,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_LOCAL,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_LOCAL,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_LOCAL,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_LOCAL,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_POTENTIAL_RESERVES_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_ADMINISTRATIVE_RESERVES_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_AUTHORIZED_RESERVES_LOCAL,UOM_VOLUME_UNIT_FACTOR,RYP_REMAINING_OVERBURDEN_VOLUME_METRICS,RYP_ANNUAL_TONNAGE_LICENSE_MAX_METRICS,RYP_ANNUAL_TONNAGE_LICENSE_AVERAGE_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_METRICS,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_METRICS,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_METRICS,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_METRICS,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_METRICS,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_METRICS,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_POTENTIAL_RESERVES_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_ADMINISTRATIVE_RESERVES_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_AUTHORIZED_RESERVES_METRICS,RYP_REMAINING_OVERBURDEN_VOLUME_CONSOLIDATED,RYP_ANNUAL_TONNAGE_LICENSE_MAX_CONSOLIDATED,RYP_ANNUAL_TONNAGE_LICENSE_AVERAGE_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_CONSOLIDATED,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_CONSOLIDATED,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_CONSOLIDATED,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_CONSOLIDATED,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_CONSOLIDATED,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_CONSOLIDATED,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_POTENTIAL_RESERVES_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_ADMINISTRATIVE_RESERVES_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_AUTHORIZED_RESERVES_CONSOLIDATED
UP901,2023,EUR,0.0,1,0.00,200000.00,0.00,0.00,0.00,0.00,1978773.00,0.00,0.00,1978773.00,0.00,0.00,9232898.00,2000000.00,1978773.00,1.00,0.00,200000.00,0.00,0.00,0.00,0.00,1978773.00,0.00,0.00,1978773.00,0.00,0.00,9232898.00,2000000.00,1978773.00,0.00,133340.00,0.00,0.00,0.00,0.00,1319247.96,0.00,0.00,1319247.96,0.00,0.00,6155573.10,1333400.00,1319247.96
UP755,2023,EUR,0.0,1,0.00,250000.00,0.00,0.00,0.00,0.00,6420770.00,0.00,0.00,6420770.00,0.00,0.00,240000.00,4225000.00,4225000.00,1.00,0.00,250000.00,0.00,0.00,0.00,0.00,6420770.00,0.00,0.00,6420770.00,0.00,0.00,240000.00,4225000.00,4225000.00,0.00,250000.00,0.00,0.00,0.00,0.00,6420770.00,0.00,0.00,6420770.00,0.00,0.00,240000.00,4225000.00,4225000.00
UP672,2023,EUR,0.0,1,0.00,0.00,0.00,4192490.00,0.00,0.00,611473.00,0.00,0.00,4803963.00,0.00,0.00,0.00,0.00,4803963.00,1.00,0.00,0.00,0.00,4192490.00,0.00,0.00,611473.00,0.00,0.00,4803963.00,0.00,0.00,0.00,0.00,4803963.00,0.00,0.00,0.00,1048122.50,0.00,0.00,152868.25,0.00,0.00,1200990.75,0.00,0.00,0.00,0.00,1200990.75
UP100,2023,EUR,0.0,1,0.00,400000.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,9000000.00,5240000.00,0.00,1.00,0.00,400000

# Construction de la table analytique finale “Fiche Carrière”

Cette étape consiste à centraliser l’ensemble des données nettoyées et enrichies dans une table analytique unique.

La table finale regroupe :
- les informations géographiques,
- les données financières,
- les volumes de production,
- les indicateurs de maintenance,
- les données climatiques,
- et les KPI stratégiques.

Cette table constitue la source principale :
- des dashboards Power BI,
- des analyses métier,
- et des business opportunities présentées lors de la soutenance.

In [0]:
from pyspark.sql import functions as F

# ============================================================
# CONFIG
# ============================================================

team_schema = "efrei_estp_2026.team6"

input_dlv = f"{team_schema}.dlv_deliveries_f_clean"
input_sites = f"{team_schema}.stc_sites_dim_clean"

output_table = f"{team_schema}.ca_annuel_par_carriere"


# ============================================================
# 1. Charger les tables nettoyées
# ============================================================

df_dlv = spark.table(input_dlv)
df_sites = spark.table(input_sites)


# ============================================================
# 2. Garder les lignes utiles pour le chiffre d'affaires
# ============================================================

df_dlv_revenue = (
    df_dlv
    .filter(F.col("STC_UP_IDENTIFIER").isNotNull())
    .filter(F.col("year").isNotNull())
    .filter(F.col("DLV_PRE_TAX_AMOUNT").isNotNull())
)


# ============================================================
# 3. Calculer le CA annuel par carrière
# ============================================================

df_ca_annuel = (
    df_dlv_revenue
    .groupBy(
        "STC_UP_IDENTIFIER",
        "year",
        "CUR_NAME"
    )
    .agg(
        F.sum("DLV_PRE_TAX_AMOUNT").alias("ca_annuel_ht"),
        F.sum("DLV_QUANTITY").alias("quantite_totale_livree"),
        F.countDistinct("DLV_NUMBER_BL").alias("nb_bons_livraison"),
        F.countDistinct("TDP_CLIENT_MDM_IDENTIFIER").alias("nb_clients"),
        F.countDistinct("PRD_IDENTIFIER").alias("nb_produits")
    )
    .withColumn(
        "prix_moyen_unitaire",
        F.when(
            F.col("quantite_totale_livree") > 0,
            F.col("ca_annuel_ht") / F.col("quantite_totale_livree")
        )
    )
)


# ============================================================
# 4. Ajouter les informations site / carrière
# ============================================================

df_ca_annuel = (
    df_ca_annuel.alias("ca")
    .join(
        df_sites.alias("site"),
        on="STC_UP_IDENTIFIER",
        how="left"
    )
    .select(
        F.col("ca.STC_UP_IDENTIFIER"),
        F.col("site.site_display_name"),
        F.col("site.STC_UP_LABEL"),
        F.col("site.STC_UP_ADDRESS_TOWN"),
        F.col("site.STC_UP_COUNTRY_ID"),
        F.col("site.STC_UP_COUNTRY_LABEL"),
        F.col("site.STC_UP_GPS_LONGITUDE"),
        F.col("site.STC_UP_GPS_LATITUDE"),
        F.col("site.site_status"),
        F.col("site.STC_UP_ACTIVITY_1_LABEL_FR"),
        F.col("ca.year"),
        F.col("ca.CUR_NAME"),
        F.col("ca.ca_annuel_ht"),
        F.col("ca.quantite_totale_livree"),
        F.col("ca.prix_moyen_unitaire"),
        F.col("ca.nb_bons_livraison"),
        F.col("ca.nb_clients"),
        F.col("ca.nb_produits")
    )
)


# ============================================================
# 5. Sauvegarder la table finale pour Power BI
# ============================================================

df_ca_annuel.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)

print(f"Table créée : {output_table}")
print("Nombre de lignes :", df_ca_annuel.count())

display(df_ca_annuel)

Table créée : efrei_estp_2026.team6.ca_annuel_par_carriere
Nombre de lignes : 6879


STC_UP_IDENTIFIER,site_display_name,STC_UP_LABEL,STC_UP_ADDRESS_TOWN,STC_UP_COUNTRY_ID,STC_UP_COUNTRY_LABEL,STC_UP_GPS_LONGITUDE,STC_UP_GPS_LATITUDE,site_status,STC_UP_ACTIVITY_1_LABEL_FR,year,CUR_NAME,ca_annuel_ht,quantite_totale_livree,prix_moyen_unitaire,nb_bons_livraison,nb_clients,nb_produits
UP343,CCO PAF CHATEAUROUX POINC,CCO PAF CHATEAUROUX POINC,LE POINCONNET,FRA,France,1.6772199869155884,46.762611389160156,Active,Recyclage,2026,EUR,43535.04999999999,5988.47,7.269811821717398,495,18,69
UP1609,COGAB-KINGUELE,COGAB-KINGUELE,KINGUELE,GAB,Gabon,null,null,Active,null,2018,(XAF),3.4632038498999983E8,28371.19000000001,12206.76274030098,880,4,13
UP2815,CCO NED TOURS SUD,CCO NED TOURS SUD,TOURS,FRA,France,0.668940007686615,47.37329864501953,Active,Dépôts pour négoce,2025,EUR,815272.69,129184.51999999999,6.310916276965692,7685,22,31
UP1160,CARRIERE DE CHASTREIX,CARRIERE DE CHASTREIX,CHASTREIX,FRA,France,2.723020076751709,45.51340866088867,Active,null,2022,EUR,137563.05000000005,28308.420000000013,4.859439347021133,735,38,19
UP3248,TERSEN-ORLY,TERSEN-ORLY,WISSOUS,FRA,France,2.342780113220215,48.732261657714844,Active,Recyclage,2022,EUR,285084.34,75539.73000000001,3.7739655675232093,496,20,17
UP2971,CMM - SITE DE TARASCON,CMM - SITE DE TARASCON,TARASCON,FRA,France,4.655280113220215,43.80057907104492,Active,null,2026,EUR,15301.359999999997,109.20000000000002,140.12234432234428,71,14,3
UP4031,CARRIERE D'ANTIGNY,CARRIERE D'ANTIGNY,ANTIGNY,FRA,France,-0.7787399888038635,46.60850143432617,Active,"Carrières, gravières",2025,EUR,4785065.34,1154800.49,4.14362946797849,18312,167,112
UP1355,CARRIERE DE CAZERES S/ L'ADOUR,CARRIERE DE CAZERES S/ L'ADOUR,CAZERES SUR ADOUR,FRA,France,-0.32714998722076416,43.764488220214844,Active,"Carrières, gravières",2020,EUR,4996667.770000001,1119872.2199999997,4.46181955473456,15901,148,50
UP1497,AP HORSENS,AP HORSENS,null,DNK,Danemark,null,null,Active,Dépôts pour négoce,2024,DKK,1.823922887E7,85010.48999999999,214.55268485101078,1689,1,14
UP522,AMV LA CHAPELLE ST LUC,AMV LA CHAPELLE ST LUC,LA CHAPELLE SAINT LUC,FRA,France,4.02016019821167,48.30678939819336,Active,null,2020,EUR,342876.87,68857.99,4.979478343762285,4298,27,12


In [0]:
from pyspark.sql import functions as F

# ============================================================
# CONFIG
# ============================================================

team_schema = "efrei_estp_2026.team6"

output_table = f"{team_schema}.fiche_carriere_bi"

# ============================================================
# 1. Charger les tables déjà créées
# ============================================================

df_ca = spark.table(f"{team_schema}.ca_annuel_par_carriere")
df_qyp = spark.table(f"{team_schema}.qyp_quarries_yearly_kpi")
df_ryp = spark.table(f"{team_schema}.ryp_quarries_reserves_yearly_kpi")
df_qib = spark.table(f"{team_schema}.qib_installations_breakdowns_yearly_kpi")

# ============================================================
# 2. Harmoniser les colonnes année
# ============================================================

# ca_annuel_par_carriere a déjà normalement une colonne "year"
df_ca = df_ca.withColumn("year", F.col("year").cast("int"))

# qyp / ryp ont souvent DAT_YEAR
if "DAT_YEAR" in df_qyp.columns:
    df_qyp = df_qyp.withColumnRenamed("DAT_YEAR", "year")

if "DAT_YEAR" in df_ryp.columns:
    df_ryp = df_ryp.withColumnRenamed("DAT_YEAR", "year")

df_qyp = df_qyp.withColumn("year", F.col("year").cast("int"))
df_ryp = df_ryp.withColumn("year", F.col("year").cast("int"))

if "year" in df_qib.columns:
    df_qib = df_qib.withColumn("year", F.col("year").cast("int"))

# ============================================================
# 3. Sélectionner uniquement les colonnes utiles
# ============================================================

ca_cols = [
    "STC_UP_IDENTIFIER",
    "year",
    "site_display_name",
    "STC_UP_LABEL",
    "STC_UP_ADDRESS_TOWN",
    "STC_UP_COUNTRY_ID",
    "STC_UP_COUNTRY_LABEL",
    "STC_UP_GPS_LONGITUDE",
    "STC_UP_GPS_LATITUDE",
    "site_status",
    "STC_UP_ACTIVITY_1_LABEL_FR",
    "CUR_NAME",
    "ca_annuel_ht",
    "quantite_totale_livree",
    "prix_moyen_unitaire",
    "nb_bons_livraison",
    "nb_clients",
    "nb_produits"
]

df_ca = df_ca.select(*[c for c in ca_cols if c in df_ca.columns])

qyp_cols = [
    "STC_UP_IDENTIFIER",
    "year",
    "production_volume",
    "QYP_PRODUCTION_VOLUME_CONSOLIDATED",
    "QYP_EXTRACTION_ANNUAL_FEES_REPORTING",
    "QYP_NET_BOOK_VALUE_FOR_DEPOSIT_LAND_REPORTING",
    "QYP_CURRENT_YEAR_LAND_DEPOSIT_DEPRECIATION_REPORTING",
    "QYP_LAND_DEPOSIT_ACQUISITION_VALUE_REPORTING",
    "extraction_cost_per_unit",
    "deposit_net_book_value_per_unit",
    "depreciation_rate",
    "remaining_book_value_rate",
    "is_active_production"
]

df_qyp = df_qyp.select(*[c for c in qyp_cols if c in df_qyp.columns])

ryp_cols = [
    "STC_UP_IDENTIFIER",
    "year",
    "reserve_volume",
    "has_reserve_data"
]

# On garde aussi toutes les colonnes de réserve détaillées si elles existent
extra_ryp_cols = [
    c for c in df_ryp.columns
    if c not in ryp_cols and c not in ["CUR_NAME", "CUR_REPORTING_NAME"]
]

df_ryp = df_ryp.select(*[c for c in ryp_cols + extra_ryp_cols if c in df_ryp.columns])

qib_cols = [
    "STC_UP_IDENTIFIER",
    "year",
    "nb_breakdown_lines",
    "total_breakdown_duration_hours",
    "avg_breakdown_duration_hours",
    "max_breakdown_duration_hours"
]

df_qib = df_qib.select(*[c for c in qib_cols if c in df_qib.columns])

# ============================================================
# 4. Joindre toutes les tables
# ============================================================

df_final = (
    df_ca.alias("ca")
    .join(
        df_qyp.alias("qyp"),
        on=["STC_UP_IDENTIFIER", "year"],
        how="left"
    )
    .join(
        df_ryp.alias("ryp"),
        on=["STC_UP_IDENTIFIER", "year"],
        how="left"
    )
    .join(
        df_qib.alias("qib"),
        on=["STC_UP_IDENTIFIER", "year"],
        how="left"
    )
)

# ============================================================
# 5. Créer des KPI finaux pour BI
# ============================================================

df_final = (
    df_final
    .withColumn(
        "reserve_life_years",
        F.when(
            (F.col("production_volume") > 0) & F.col("reserve_volume").isNotNull(),
            F.col("reserve_volume") / F.col("production_volume")
        )
    )
    .withColumn(
        "availability_proxy",
        F.when(
            F.col("total_breakdown_duration_hours").isNotNull(),
            1 - (F.col("total_breakdown_duration_hours") / F.lit(365 * 24))
        )
    )
    .withColumn(
        "ca_per_client",
        F.when(
            F.col("nb_clients") > 0,
            F.col("ca_annuel_ht") / F.col("nb_clients")
        )
    )
    .withColumn(
        "ca_per_product",
        F.when(
            F.col("nb_produits") > 0,
            F.col("ca_annuel_ht") / F.col("nb_produits")
        )
    )
)

# ============================================================
# 6. Nettoyage final des KPI
# ============================================================

df_final = df_final.withColumn(
    "availability_proxy",
    F.when(F.col("availability_proxy") < 0, F.lit(0))
     .when(F.col("availability_proxy") > 1, F.lit(1))
     .otherwise(F.col("availability_proxy"))
)

# ============================================================
# 7. Sauvegarder la table finale
# ============================================================

df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)

print(f"Table finale créée : {output_table}")
print("Nombre de lignes :", df_final.count())
print("Nombre de carrières :", df_final.select("STC_UP_IDENTIFIER").distinct().count())

display(df_final)

Table finale créée : efrei_estp_2026.team6.fiche_carriere_bi
Nombre de lignes : 6879
Nombre de carrières : 1172


STC_UP_IDENTIFIER,year,site_display_name,STC_UP_LABEL,STC_UP_ADDRESS_TOWN,STC_UP_COUNTRY_ID,STC_UP_COUNTRY_LABEL,STC_UP_GPS_LONGITUDE,STC_UP_GPS_LATITUDE,site_status,STC_UP_ACTIVITY_1_LABEL_FR,CUR_NAME,ca_annuel_ht,quantite_totale_livree,prix_moyen_unitaire,nb_bons_livraison,nb_clients,nb_produits,production_volume,QYP_PRODUCTION_VOLUME_CONSOLIDATED,QYP_EXTRACTION_ANNUAL_FEES_REPORTING,QYP_NET_BOOK_VALUE_FOR_DEPOSIT_LAND_REPORTING,QYP_CURRENT_YEAR_LAND_DEPOSIT_DEPRECIATION_REPORTING,extraction_cost_per_unit,deposit_net_book_value_per_unit,is_active_production,reserve_volume,has_reserve_data,RYP_REMAINING_OVERBURDEN_VOLUME_LOCAL,RYP_ANNUAL_TONNAGE_LICENSE_MAX_LOCAL,RYP_ANNUAL_TONNAGE_LICENSE_AVERAGE_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_LOCAL,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_LOCAL,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_LOCAL,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_LOCAL,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_LOCAL,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_LOCAL,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_POTENTIAL_RESERVES_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_ADMINISTRATIVE_RESERVES_LOCAL,RYP_END_OF_YEAR_QUANTITY_ON_AUTHORIZED_RESERVES_LOCAL,UOM_VOLUME_UNIT_FACTOR,RYP_REMAINING_OVERBURDEN_VOLUME_METRICS,RYP_ANNUAL_TONNAGE_LICENSE_MAX_METRICS,RYP_ANNUAL_TONNAGE_LICENSE_AVERAGE_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_METRICS,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_METRICS,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_METRICS,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_METRICS,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_METRICS,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_METRICS,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_POTENTIAL_RESERVES_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_ADMINISTRATIVE_RESERVES_METRICS,RYP_END_OF_YEAR_QUANTITY_ON_AUTHORIZED_RESERVES_METRICS,RYP_REMAINING_OVERBURDEN_VOLUME_CONSOLIDATED,RYP_ANNUAL_TONNAGE_LICENSE_MAX_CONSOLIDATED,RYP_ANNUAL_TONNAGE_LICENSE_AVERAGE_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_CONSOLIDATED,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_CONSOLIDATED,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_PROPERTY_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_CONSOLIDATED,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_CONSOLIDATED,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_DRILLING_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_CONSOLIDATED,RYP_DEPOSIT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_CONSOLIDATED,RYP_ADJUSTMENT_EXTRACTION_QUANTITY_ON_PHYSICAL_RESERVES_TOTAL_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_POTENTIAL_RESERVES_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_ADMINISTRATIVE_RESERVES_CONSOLIDATED,RYP_END_OF_YEAR_QUANTITY_ON_AUTHORIZED_RESERVES_CONSOLIDATED,nb_breakdown_lines,total_breakdown_duration_hours,avg_breakdown_duration_hours,max_breakdown_duration_hours,reserve_life_years,availability_proxy,ca_per_client,ca_per_product
UP3705,2024,CSO ANGOU ROULLET ST ESTEPHE,CSO ANGOU ROULLET ST ESTEPHE,ROULLET SAINT ESTEPHE,FRA,France,0.07963000237941742,45.60517120361328,Active,Décharges,EUR,1015329.98,165696.21,6.127659648944294,10875,183,92,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null

In [0]:
from pyspark.sql import functions as F

team_schema = "efrei_estp_2026.team6"
output_table = f"{team_schema}.fiche_carriere_bi"

df_ca = spark.table(f"{team_schema}.ca_annuel_par_carriere")
df_qyp = spark.table(f"{team_schema}.qyp_quarries_yearly_kpi")
df_ryp = spark.table(f"{team_schema}.ryp_quarries_reserves_yearly_kpi")
df_qib = spark.table(f"{team_schema}.qib_installations_breakdowns_yearly_kpi")

# Harmonisation année
df_ca = df_ca.withColumn("year", F.col("year").cast("int"))

if "DAT_YEAR" in df_qyp.columns:
    df_qyp = df_qyp.withColumnRenamed("DAT_YEAR", "year")
df_qyp = df_qyp.withColumn("year", F.col("year").cast("int"))

if "DAT_YEAR" in df_ryp.columns:
    df_ryp = df_ryp.withColumnRenamed("DAT_YEAR", "year")
df_ryp = df_ryp.withColumn("year", F.col("year").cast("int"))

df_qib = df_qib.withColumn("year", F.col("year").cast("int"))

# Sélection colonnes utiles
ca_cols = [
    "STC_UP_IDENTIFIER", "year", "site_display_name", "STC_UP_LABEL",
    "STC_UP_ADDRESS_TOWN", "STC_UP_COUNTRY_ID", "STC_UP_COUNTRY_LABEL",
    "STC_UP_GPS_LONGITUDE", "STC_UP_GPS_LATITUDE", "site_status",
    "STC_UP_ACTIVITY_1_LABEL_FR", "CUR_NAME", "ca_annuel_ht",
    "quantite_totale_livree", "prix_moyen_unitaire", "nb_bons_livraison",
    "nb_clients", "nb_produits"
]

qyp_cols = [
    "STC_UP_IDENTIFIER", "year", "production_volume",
    "QYP_PRODUCTION_VOLUME_CONSOLIDATED",
    "QYP_EXTRACTION_ANNUAL_FEES_REPORTING",
    "QYP_NET_BOOK_VALUE_FOR_DEPOSIT_LAND_REPORTING",
    "QYP_CURRENT_YEAR_LAND_DEPOSIT_DEPRECIATION_REPORTING",
    "QYP_LAND_DEPOSIT_ACQUISITION_VALUE_REPORTING",
    "extraction_cost_per_unit", "deposit_net_book_value_per_unit",
    "depreciation_rate", "remaining_book_value_rate",
    "is_active_production"
]

ryp_cols = [
    "STC_UP_IDENTIFIER", "year", "reserve_volume", "has_reserve_data"
]

qib_cols = [
    "STC_UP_IDENTIFIER", "year", "nb_breakdown_lines",
    "total_breakdown_duration_hours",
    "avg_breakdown_duration_hours",
    "max_breakdown_duration_hours"
]

df_ca = df_ca.select(*[c for c in ca_cols if c in df_ca.columns])
df_qyp = df_qyp.select(*[c for c in qyp_cols if c in df_qyp.columns])
df_ryp = df_ryp.select(*[c for c in ryp_cols if c in df_ryp.columns])
df_qib = df_qib.select(*[c for c in qib_cols if c in df_qib.columns])

# Jointure finale
df_final = (
    df_ca
    .join(df_qyp, on=["STC_UP_IDENTIFIER", "year"], how="left")
    .join(df_ryp, on=["STC_UP_IDENTIFIER", "year"], how="left")
    .join(df_qib, on=["STC_UP_IDENTIFIER", "year"], how="left")
)

# KPI finaux
df_final = (
    df_final
    .withColumn(
        "reserve_life_years",
        F.when(
            (F.col("production_volume") > 0) & F.col("reserve_volume").isNotNull(),
            F.col("reserve_volume") / F.col("production_volume")
        )
    )
    .withColumn(
        "availability_proxy",
        F.when(
            F.col("total_breakdown_duration_hours").isNotNull(),
            1 - (F.col("total_breakdown_duration_hours") / F.lit(365 * 24))
        )
    )
    .withColumn(
        "availability_proxy",
        F.when(F.col("availability_proxy") < 0, F.lit(0))
         .when(F.col("availability_proxy") > 1, F.lit(1))
         .otherwise(F.col("availability_proxy"))
    )
    .withColumn(
        "ca_per_client",
        F.when(F.col("nb_clients") > 0, F.col("ca_annuel_ht") / F.col("nb_clients"))
    )
    .withColumn(
        "ca_per_product",
        F.when(F.col("nb_produits") > 0, F.col("ca_annuel_ht") / F.col("nb_produits"))
    )
)

df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)

print(f"Table créée : {output_table}")
print("Lignes :", df_final.count())
display(df_final)

Table créée : efrei_estp_2026.team6.fiche_carriere_bi
Lignes : 6879


STC_UP_IDENTIFIER,year,site_display_name,STC_UP_LABEL,STC_UP_ADDRESS_TOWN,STC_UP_COUNTRY_ID,STC_UP_COUNTRY_LABEL,STC_UP_GPS_LONGITUDE,STC_UP_GPS_LATITUDE,site_status,STC_UP_ACTIVITY_1_LABEL_FR,CUR_NAME,ca_annuel_ht,quantite_totale_livree,prix_moyen_unitaire,nb_bons_livraison,nb_clients,nb_produits,production_volume,QYP_PRODUCTION_VOLUME_CONSOLIDATED,QYP_EXTRACTION_ANNUAL_FEES_REPORTING,QYP_NET_BOOK_VALUE_FOR_DEPOSIT_LAND_REPORTING,QYP_CURRENT_YEAR_LAND_DEPOSIT_DEPRECIATION_REPORTING,extraction_cost_per_unit,deposit_net_book_value_per_unit,is_active_production,reserve_volume,has_reserve_data,nb_breakdown_lines,total_breakdown_duration_hours,avg_breakdown_duration_hours,max_breakdown_duration_hours,reserve_life_years,availability_proxy,ca_per_client,ca_per_product
UP3705,2024,CSO ANGOU ROULLET ST ESTEPHE,CSO ANGOU ROULLET ST ESTEPHE,ROULLET SAINT ESTEPHE,FRA,France,0.07963000237941742,45.60517120361328,Active,Décharges,EUR,1015329.98,165696.21,6.127659648944294,10875,183,92,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,5548.251256830601,11036.195434782609
UP898,2026,MIDI-CONCASSAGE - ENTRESSEN,MIDI-CONCASSAGE - ENTRESSEN,ENTRESSEN,FRA,France,4.974400043487549,43.585941314697266,Active,"Carrières, gravières",EUR,606986.6400000001,128804.61999999995,4.712460158649591,4040,113,59,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,5371.563185840709,10287.909152542376
UP1218,2025,SITE DE LAROQUE D'OLMES,SITE DE LAROQUE D'OLMES,LAROQUES D OLMES,FRA,France,1.8653600215911865,42.96659851074219,Active,Décharges,EUR,23564.309999999998,3746.9999999999995,6.28884707766213,220,28,18,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,841.5824999999999,1309.128333333333
UP2805,2022,CCO NED VANNES PLOEREN,CCO NED VANNES PLOEREN,PLOEREN,FRA,France,-2.867029905319214,47.65549850463867,Active,Dépôts pour négoce,EUR,234371.52,39671.41,5.907819258251722,3308,68,41,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3446.64,5716.3785365853655
UP586,2022,SVPR SAINTE MARGUERITE,SVPR SAINTE MARGUERITE,SAINTE-MARGUERITE,FRA,France,6.972149848937988,48.26620864868164,Active,Postes d'enrobage à chaud,EUR,400651.62999999995,15176.640000000001,26.399231318658142,21,4,6,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,100162.90749999999,66775.27166666665
UP378,2021,CCO NED MEUNIER,CCO NED MEUNIER,NOGENT SUR VERNISSON,FRA,France,2.746220111846924,47.85049819946289,Active,null,EUR,57332.86999999999,103.76999999999997,552.4994699816904,66,5,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,11466.573999999997,19110.95666666666
UP608,2021,CIDFN-PLATEFORME ETAMPES,CIDFN-PLATEFORME ETAMPES,ETAMPES,FRA,France,2.171020030975342,48.45109939575195,Active,Industries Béton BPE,EUR,1676415.4099999995,145440.57,11.526463420763541,11222,112,156,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,14967.994732142852,10746.252628205124
UP4023,2024,CARRIERE DE LA PEYRATTE,CARRIERE DE LA PEYRATTE,LA PEYRATTE,FRA,France,-0.16997000575065613,46.67108154296875,Active,"Carrières, gravières",EUR,8307130.690000003,2430202.6599999974,3.4182872180709456,24774,242,114,611008.00,611008.00,176494.00,6076835.00,253111.00,0.2888571017073,9.9455899104431,1,0.0,1,null,null,null,null,0.0,null,34326.986322314064,72869.56745614037
UP567,2020,CARRIERE DE PERIGNY,CARRIERE DE PERIGNY,PERIGNY LA ROSE,FRA,France,3.6236701011657715,48.55691909790039,Active,"Carrières, gravières",null,6468353.870000007,2333615.12,2.771816918121445,25014,80,19,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,80854.42337500009,340439.6773684214
UP3082,2024,CSO AG NEGOCE MONTAUBAN,CSO AG NEGOCE MONTAUBAN,MERIGNAC,FRA,France,-0.692300021648407,44.8366813659668,Active,Dépôts pour négoce,EUR,201820.91,2131.75,94.67381728626715,126,11,9,null,null,null,null,null,null,null,null,null,null,null

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

team_schema = "efrei_estp_2026.team6"

input_table = f"{team_schema}.fiche_carriere_bi"
output_table = f"{team_schema}.carriere_score_bi"

df = spark.table(input_table)

w = Window.partitionBy("year")

def minmax_score(col_name):
    return F.when(
        F.max(F.col(col_name)).over(w) != F.min(F.col(col_name)).over(w),
        (F.col(col_name) - F.min(F.col(col_name)).over(w)) /
        (F.max(F.col(col_name)).over(w) - F.min(F.col(col_name)).over(w))
    )

df_score = (
    df
    .withColumn("revenue_score", minmax_score("ca_annuel_ht"))
    .withColumn("volume_score", minmax_score("quantite_totale_livree"))
    .withColumn("production_score", minmax_score("production_volume"))
    .withColumn("reserve_score", minmax_score("reserve_life_years"))
    .withColumn("availability_score", minmax_score("availability_proxy"))
)

# Pour les pannes : plus il y en a, plus le score doit être mauvais
df_score = df_score.withColumn(
    "breakdown_score",
    F.when(
        F.col("nb_breakdown_lines").isNotNull(),
        1 - minmax_score("nb_breakdown_lines")
    )
)

# Remplacer les scores manquants par 0.5 pour ne pas pénaliser trop fort les données absentes
score_cols = [
    "revenue_score",
    "volume_score",
    "production_score",
    "reserve_score",
    "availability_score",
    "breakdown_score"
]

for c in score_cols:
    if c in df_score.columns:
        df_score = df_score.withColumn(c, F.coalesce(F.col(c), F.lit(0.5)))

df_score = (
    df_score
    .withColumn(
        "global_health_score",
        (
            F.col("revenue_score") * 0.25 +
            F.col("volume_score") * 0.15 +
            F.col("production_score") * 0.15 +
            F.col("reserve_score") * 0.20 +
            F.col("availability_score") * 0.15 +
            F.col("breakdown_score") * 0.10
        )
    )
    .withColumn(
        "business_priority",
        F.when(F.col("global_health_score") >= 0.75, F.lit("Leader"))
         .when(F.col("global_health_score") >= 0.55, F.lit("Stable"))
         .when(F.col("global_health_score") >= 0.35, F.lit("À surveiller"))
         .otherwise(F.lit("Priorité d'action"))
    )
)

cols_score = [
    "STC_UP_IDENTIFIER", "year", "site_display_name",
    "STC_UP_ADDRESS_TOWN", "STC_UP_COUNTRY_LABEL",
    "STC_UP_GPS_LONGITUDE", "STC_UP_GPS_LATITUDE",
    "ca_annuel_ht", "quantite_totale_livree", "production_volume",
    "reserve_volume", "reserve_life_years", "nb_breakdown_lines",
    "availability_proxy",
    "revenue_score", "volume_score", "production_score",
    "reserve_score", "availability_score", "breakdown_score",
    "global_health_score", "business_priority"
]

df_score = df_score.select(*[c for c in cols_score if c in df_score.columns])

df_score.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)

print(f"Table créée : {output_table}")
display(df_score)

Table créée : efrei_estp_2026.team6.carriere_score_bi


STC_UP_IDENTIFIER,year,site_display_name,STC_UP_ADDRESS_TOWN,STC_UP_COUNTRY_LABEL,STC_UP_GPS_LONGITUDE,STC_UP_GPS_LATITUDE,ca_annuel_ht,quantite_totale_livree,production_volume,reserve_volume,reserve_life_years,nb_breakdown_lines,availability_proxy,revenue_score,volume_score,production_score,reserve_score,availability_score,breakdown_score,global_health_score,business_priority
UP1860,2012,CARRIERE KOUNGOU,KOUNGOU,France,45.19142150878906,-12.736080169677734,128.56,9.11,null,null,null,null,null,0.5,0.5,0.5,0.5,0.5,0.5,0.5,À surveiller
UP1860,2013,CARRIERE KOUNGOU,KOUNGOU,France,45.19142150878906,-12.736080169677734,8847.16,563.29,null,null,null,null,null,0.2142073539078758,0.0,0.5,0.5,0.5,0.5,0.3535518384769689,À surveiller
UP1861,2013,CARRIERE PAMANDZI,PAMANDZI,France,45.2830696105957,-12.796299934387207,23305.64,8549.72,null,null,null,null,null,0.5642759343709786,0.6601080945260925,0.5,0.5,0.5,0.5,0.5400851977716585,À surveiller
UP2883,2013,DEPOT HANDREMA,HANDREMA,France,45.09877014160156,-12.680769920349121,12302.09,2669.0799999999995,null,null,null,null,null,0.2978580862600586,0.17405136267044227,0.5,0.5,0.5,0.5,0.40057222596558095,À surveiller
UP2880,2013,DEPOT COMBANI,COMBANI,France,45.13636016845703,-12.788339614868164,18977.350000000002,7722.130000000001,null,null,null,null,null,0.45947941799217235,0.5917047080381564,0.5,0.5,0.5,0.5,0.5036255607037665,À surveiller
UP2882,2013,CARRIERE ILONI,ILONI,France,45.191009521484375,-12.846380233764648,7292.9400000000005,2019.07,null,null,null,null,null,0.17657659402665984,0.12032562256843109,0.5,0.5,0.5,0.5,0.3621929918919296,À surveiller
UP1863,2013,MAJICAVO,MAJICAVO,France,45.219398498535156,-12.748029708862305,16387.01,6505.28,null,null,null,null,null,0.3967621305098924,0.49112753715904317,0.5,0.5,0.5,0.5,0.47285966320132955,À surveiller
UP1862,2013,CARRIERE M TSAMOUDOU,MTSAMOUDOU,France,45.214271545410156,-12.796270370483398,41301.85,12661.96,null,null,null,null,null,1.0,1.0,0.5,0.5,0.5,0.5,0.7000000000000001,Stable
UP2881,2013,DEPOT LONGONI,LONGONI,France,45.16429138183594,-12.733570098876953,2926.36,1564.6,null,null,null,null,null,0.0708530005314532,0.08276198954099914,0.5,0.5,0.5,0.5,0.33012754856401316,Priorité d'action
UP589,2013,null,null,null,null,null,0.0,698.99,null,null,null,null,null,0.0,0.0112161088780833,0.5,0.5,0.5,0.5,0.3016824163317125,Priorité d'action


In [0]:
from pyspark.sql import functions as F

# ============================================================
# CONFIG
# ============================================================

team_schema = "efrei_estp_2026.team6"

input_product_sales = f"{team_schema}.dlv_product_yearly_kpi"
input_products = f"{team_schema}.prd_products_dim_clean"
input_ca = f"{team_schema}.ca_annuel_par_carriere"

output_table = f"{team_schema}.product_mix_bi"

# ============================================================
# 1. CHARGER LES TABLES
# ============================================================

df_sales = spark.table(input_product_sales)
df_products = spark.table(input_products)
df_ca = spark.table(input_ca)

df_sales = df_sales.withColumn("year", F.col("year").cast("int"))
df_ca = df_ca.withColumn("year", F.col("year").cast("int"))

# ============================================================
# 2. RENOMMER LES COLONNES PRODUIT POUR ÉVITER LES AMBIGUÏTÉS
# ============================================================

df_sales = (
    df_sales
    .withColumnRenamed("PRD_IDENTIFIER", "sales_PRD_IDENTIFIER")
)

if "PRD_IDENTIFIER" in df_products.columns:
    df_products = df_products.withColumnRenamed("PRD_IDENTIFIER", "dim_PRD_IDENTIFIER")

# ============================================================
# 3. JOINDRE LES VENTES AVEC LA DIMENSION PRODUIT
# ============================================================

if "PRD_KEY" in df_sales.columns and "PRD_KEY" in df_products.columns:
    df_mix = df_sales.join(df_products, on="PRD_KEY", how="left")
else:
    df_mix = df_sales.join(
        df_products,
        df_sales["sales_PRD_IDENTIFIER"] == df_products["dim_PRD_IDENTIFIER"],
        how="left"
    )

# ============================================================
# 4. AJOUTER LE CA ANNUEL DU SITE
# ============================================================

df_ca_small = (
    df_ca
    .select(
        "STC_UP_IDENTIFIER",
        "year",
        "site_display_name",
        "ca_annuel_ht"
    )
    .dropDuplicates(["STC_UP_IDENTIFIER", "year"])
)

df_mix = df_mix.join(
    df_ca_small,
    on=["STC_UP_IDENTIFIER", "year"],
    how="left"
)

# ============================================================
# 5. CHOISIR LA COLONNE DE GROUPE PRODUIT
# ============================================================

if "product_family" in df_mix.columns:
    product_group_col = "product_family"
elif "PRD_TYPE" in df_mix.columns:
    product_group_col = "PRD_TYPE"
else:
    product_group_col = "sales_PRD_IDENTIFIER"

# ============================================================
# 6. CRÉER LA TABLE PRODUCT MIX
# ============================================================

df_product_mix = (
    df_mix
    .groupBy(
        "STC_UP_IDENTIFIER",
        "year",
        "site_display_name",
        product_group_col
    )
    .agg(
        F.sum("product_quantity").alias("product_quantity"),
        F.sum("product_revenue_ht").alias("product_revenue_ht"),
        F.avg("avg_product_price").alias("avg_product_price"),
        F.max("ca_annuel_ht").alias("site_revenue_ht"),
        F.countDistinct("sales_PRD_IDENTIFIER").alias("nb_product_codes")
    )
    .withColumn(
        "share_of_site_revenue",
        F.when(
            F.col("site_revenue_ht") > 0,
            F.col("product_revenue_ht") / F.col("site_revenue_ht")
        )
    )
)

# Renommer la colonne de famille produit avec un nom stable
if product_group_col != "product_family":
    df_product_mix = df_product_mix.withColumnRenamed(product_group_col, "product_family")

# ============================================================
# 7. SAUVEGARDER
# ============================================================

df_product_mix.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)

print(f"Table créée : {output_table}")
print("Lignes :", df_product_mix.count())

display(df_product_mix)

Table créée : efrei_estp_2026.team6.product_mix_bi
Lignes : 17541


STC_UP_IDENTIFIER,year,site_display_name,product_family,product_quantity,product_revenue_ht,avg_product_price,site_revenue_ht,nb_product_codes,share_of_site_revenue
UP597,2023,CIDFN-CASES ST VIGOR D YMONVIL,OTHER,85184.46,647263.5900000001,89.52199693956808,1086147.42,25,0.5959260944522615
UP630,2022,TERSEN-PLATEFORME PIERRELAYE,OTHER,129524.91000000003,4398152.939999999,40.161176122009124,5341347.170000006,227,0.8234164153759721
UP4058,2024,CARRIERE DE GREZAC,CONCRETE,4.16,134.2,32.25961538461538,2853392.3900000015,1,4.703173684429708E-5
UP1081,2019,CARRIERE D'ALLENC,OTHER,18347.950000000008,119362.80999999998,6.123025499733225,136528.88999999998,7,0.8742677831776117
UP630,2024,TERSEN-PLATEFORME PIERRELAYE,OTHER,133159.23999999996,4171406.09,44.686568752236106,5306566.140000005,206,0.7860838779633105
UP4058,2023,CARRIERE DE GREZAC,OTHER,423905.18,1825323.5299999996,30.354570860476972,2568009.8299999996,45,0.7107930463023188
UP3100,2020,SITE DE CHAUVE,OTHER,168463.72999999995,993627.22,9.481400632601988,1626637.6299999997,29,0.6108473096125289
UP3100,2021,SITE DE CHAUVE,ASPHALT,213.17,26127.190000000002,122.56504198526999,1720265.6100000003,1,0.015187881364436506
UP630,2022,TERSEN-PLATEFORME PIERRELAYE,GRAVEL,35520.73999999998,476480.72000000003,35.259672331712714,5341347.170000006,15,0.08920609442430222
UP1134,2021,ETABLISSEMENT DE LEMPDES,OTHER,7345.969999999999,119403.87000000001,15.59822443102258,313681.62999999995,2,0.38065305258710885


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ============================================================
# CONFIG
# ============================================================

team_schema = "efrei_estp_2026.team6"

input_table = "efrei_estp_2026.team6.ecoact_data_clean"  # à remplacer
input_sites = f"{team_schema}.stc_sites_dim_clean"

output_detail = f"{team_schema}.climate_risks_bi"
output_summary = f"{team_schema}.climate_risks_site_summary_bi"

# ============================================================
# 1. CHARGER LES TABLES
# ============================================================

df = spark.table(input_table)
df_sites = spark.table(input_sites)

print("Lignes EcoAct initiales :", df.count())
print("Colonnes EcoAct initiales :", len(df.columns))

# ============================================================
# 2. NETTOYER LES CHAÎNES VIDES
# ============================================================

empty_values = [
    "", "-", "(VIDE)", "(EMPTY)", "null", "NULL",
    "[BLANK]", "N/A", "NA", "nan", "NaN", "n.a.", "n.a"
]

string_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, StringType)
]

for c in string_cols:
    df = df.withColumn(
        c,
        F.when(F.trim(F.col(c)).isin(empty_values), F.lit(None))
         .otherwise(F.trim(F.col(c)))
    )

# ============================================================
# 3. SUPPRIMER LES COLONNES INUTILES / DOUBLONS NORMALISÉS
# ============================================================

cols_to_drop = [
    c for c in df.columns
    if c.endswith("_NORMALIZED") or c == "_C29"
]

df = df.drop(*cols_to_drop)

print("Colonnes supprimées :", cols_to_drop)

# ============================================================
# 4. FONCTION POUR CONVERTIR LES NOMBRES AVEC VIRGULE
# ============================================================

def to_double_fr(col_name):
    return (
        F.when(
            F.col(col_name).isNull(),
            F.lit(None).cast("double")
        )
        .otherwise(
            F.regexp_replace(F.col(col_name).cast("string"), ",", ".").cast("double")
        )
    )

numeric_cols = [
    "LAT",
    "LON",
    "PARAMETER",
    "REFERENCE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS",
    "FUTURE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS",
    "REFERENCE_PERIOD_YEAR_STD_MEDIAN_OF_MODELS",
    "FUTURE_PERIOD_YEAR_STD_MEDIAN_OF_MODELS",
    "NUMBER_OF_MODELS",
    "ANOMALY_MEAN_FUTURE_MINUS_REF",
    "ANOMALY_STD_FUTURE_MINUS_REF",
    "REFERENCE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_RAW",
    "FUTURE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_RAW"
]

for c in numeric_cols:
    if c in df.columns:
        df = df.withColumn(c, to_double_fr(c))

# ============================================================
# 5. REMPLACER -999999 PAR NULL POUR LES COLONNES NUMÉRIQUES
# ============================================================

for c in numeric_cols:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.when(F.col(c) == -999999, F.lit(None).cast("double"))
             .otherwise(F.col(c))
        )

# ============================================================
# 6. RENOMMER LES COLONNES POUR POWER BI
# ============================================================

rename_map = {
    "SITE_ID": "ecoact_site_id",
    "CLIENT_NAME": "client_name",
    "LAT": "climate_latitude",
    "LON": "climate_longitude",
    "INDICATOR": "climate_indicator",
    "UNIT": "climate_unit",
    "PARAMETER": "climate_parameter",
    "PARAMETER_UNIT": "climate_parameter_unit",
    "SCENARIO": "climate_scenario",
    "REFERENCE_PERIOD": "reference_period",
    "FUTURE_PERIOD": "future_period",
    "REFERENCE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS": "reference_mean",
    "FUTURE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS": "future_mean",
    "NUMBER_OF_MODELS": "number_of_models",
    "MODEL_CONFIDENCE": "model_confidence",
    "ANOMALY_MEAN_FUTURE_MINUS_REF": "climate_anomaly",
    "REFERENCE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_RAW": "reference_exposure_raw",
    "REFERENCE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_LABEL": "reference_exposure_label",
    "FUTURE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_RAW": "future_exposure_raw",
    "FUTURE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_LABEL": "future_exposure_label",
    "SITE_NAME": "ecoact_site_name",
    "SITE_TYPOLOGY": "site_typology",
    "STRUCTURE_CODE": "structure_code",
    "LIBELLÉ_STRUCTURE": "structure_label",
    "STRUCTURE_ACTIVITE": "structure_activity",
    "EU_TAXONOMY_HAZARD": "eu_taxonomy_hazard"
}

for old, new in rename_map.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)

# ============================================================
# 7. CRÉER UNE CLÉ DE JOINTURE AVEC LES SITES
# ============================================================

# STRUCTURE_CODE contient parfois des UPxxxx ou UExxxx.
# Pour joindre à stc_sites_dim_clean, on commence par les codes UP.
df = df.withColumn(
    "STC_UP_IDENTIFIER",
    F.when(F.upper(F.col("structure_code")).startswith("UP"), F.col("structure_code"))
)

# Jointure avec dimension sites si possible
df = (
    df.alias("clim")
    .join(
        df_sites.select(
            "STC_UP_IDENTIFIER",
            "site_display_name",
            "STC_UP_LABEL",
            "STC_UP_COUNTRY_LABEL",
            "STC_UP_ADDRESS_TOWN",
            "STC_UP_GPS_LONGITUDE",
            "STC_UP_GPS_LATITUDE",
            "STC_UP_ACTIVITY_1_LABEL_FR"
        ).alias("site"),
        on="STC_UP_IDENTIFIER",
        how="left"
    )
)

# ============================================================
# 8. CRÉER UN SCORE DE RISQUE CLIMATIQUE
# ============================================================

df = df.withColumn(
    "future_exposure_level_clean",
    F.lower(F.col("future_exposure_label"))
)

df = df.withColumn(
    "climate_risk_score",
    F.when(F.col("future_exposure_level_clean").isin("extreme"), F.lit(5))
     .when(F.col("future_exposure_level_clean").isin("very high"), F.lit(4))
     .when(F.col("future_exposure_level_clean").isin("high", "medium - high"), F.lit(3))
     .when(F.col("future_exposure_level_clean").isin("medium", "low - medium"), F.lit(2))
     .when(F.col("future_exposure_level_clean").isin("low", "very low"), F.lit(1))
     .otherwise(F.lit(None).cast("int"))
)

# Catégories simplifiées
df = df.withColumn(
    "climate_hazard_family",
    F.when(F.lower(F.col("climate_indicator")).rlike("heat|cooling"), F.lit("Heat"))
     .when(F.lower(F.col("climate_indicator")).rlike("fire|wildfire"), F.lit("Fire"))
     .when(F.lower(F.col("climate_indicator")).rlike("drought|dry"), F.lit("Drought"))
     .when(F.lower(F.col("climate_indicator")).rlike("flood|water|sea|saline"), F.lit("Water"))
     .when(F.lower(F.col("climate_indicator")).rlike("wind|tornado"), F.lit("Wind"))
     .when(F.lower(F.col("climate_indicator")).rlike("hail|wet"), F.lit("Storm / Hail"))
     .when(F.lower(F.col("climate_indicator")).rlike("soil|solifluction"), F.lit("Soil"))
     .otherwise(F.lit("Other"))
)

df = df.withColumn(
    "is_fire_risk",
    F.when(
        F.lower(F.coalesce(F.col("climate_indicator"), F.lit(""))).rlike("fire|wildfire|incendie")
        | F.lower(F.coalesce(F.col("eu_taxonomy_hazard"), F.lit(""))).rlike("fire|wildfire|incendie"),
        F.lit(1)
    ).otherwise(F.lit(0))
)

# ============================================================
# 9. SÉLECTIONNER LES COLONNES BI
# ============================================================

cols_detail = [
    "STC_UP_IDENTIFIER",
    "site_display_name",
    "STC_UP_LABEL",
    "STC_UP_ADDRESS_TOWN",
    "STC_UP_COUNTRY_LABEL",
    "STC_UP_GPS_LONGITUDE",
    "STC_UP_GPS_LATITUDE",
    "ecoact_site_id",
    "ecoact_site_name",
    "climate_latitude",
    "climate_longitude",
    "structure_code",
    "structure_label",
    "structure_activity",
    "site_typology",
    "climate_indicator",
    "climate_hazard_family",
    "eu_taxonomy_hazard",
    "climate_scenario",
    "reference_period",
    "future_period",
    "model_confidence",
    "number_of_models",
    "reference_mean",
    "future_mean",
    "climate_anomaly",
    "reference_exposure_label",
    "future_exposure_label",
    "future_exposure_raw",
    "climate_risk_score",
    "is_fire_risk"
]

cols_detail_existing = [c for c in cols_detail if c in df.columns]

df_climate_bi = df.select(*cols_detail_existing)

# ============================================================
# 10. CRÉER TABLE RÉSUMÉE PAR SITE
# ============================================================

df_summary = (
    df_climate_bi
    .groupBy("STC_UP_IDENTIFIER")
    .agg(
        F.max("climate_risk_score").alias("max_climate_risk_score"),
        F.avg("climate_risk_score").alias("avg_climate_risk_score"),
        F.sum("is_fire_risk").alias("nb_fire_risk_lines"),
        F.countDistinct("climate_indicator").alias("nb_climate_indicators"),
        F.countDistinct("climate_hazard_family").alias("nb_hazard_families"),
        F.first("site_display_name", ignorenulls=True).alias("site_display_name"),
        F.first("STC_UP_COUNTRY_LABEL", ignorenulls=True).alias("STC_UP_COUNTRY_LABEL"),
        F.first("STC_UP_ADDRESS_TOWN", ignorenulls=True).alias("STC_UP_ADDRESS_TOWN"),
        F.first("STC_UP_GPS_LONGITUDE", ignorenulls=True).alias("STC_UP_GPS_LONGITUDE"),
        F.first("STC_UP_GPS_LATITUDE", ignorenulls=True).alias("STC_UP_GPS_LATITUDE")
    )
    .withColumn(
        "climate_risk_level",
        F.when(F.col("max_climate_risk_score") >= 5, F.lit("Extreme"))
         .when(F.col("max_climate_risk_score") >= 4, F.lit("Very high"))
         .when(F.col("max_climate_risk_score") >= 3, F.lit("High"))
         .when(F.col("max_climate_risk_score") >= 2, F.lit("Medium"))
         .when(F.col("max_climate_risk_score") >= 1, F.lit("Low"))
         .otherwise(F.lit("Unknown"))
    )
    .withColumn(
        "has_fire_risk",
        F.when(F.col("nb_fire_risk_lines") > 0, F.lit(1)).otherwise(F.lit(0))
    )
)

# ============================================================
# 11. SAUVEGARDE
# ============================================================

df_climate_bi.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_detail)

df_summary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_summary)

print("Tables créées :")
print("-", output_detail)
print("-", output_summary)

print("Lignes détail :", df_climate_bi.count())
print("Lignes résumé :", df_summary.count())

display(df_climate_bi)
display(df_summary)

Lignes EcoAct initiales : 120858
Colonnes EcoAct initiales : 57
Colonnes supprimées : ['_C29', 'CLIENT_NAME_NORMALIZED', 'LAT_NORMALIZED', 'LON_NORMALIZED', 'INDICATOR_NORMALIZED', 'UNIT_NORMALIZED', 'PARAMETER_NORMALIZED', 'PARAMETER_UNIT_NORMALIZED', 'SCENARIO_NORMALIZED', 'REFERENCE_PERIOD_NORMALIZED', 'FUTURE_PERIOD_NORMALIZED', 'REFERENCE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS_NORMALIZED', 'FUTURE_PERIOD_YEAR_MEAN_MEDIAN_OF_MODELS_NORMALIZED', 'REFERENCE_PERIOD_YEAR_STD_MEDIAN_OF_MODELS_NORMALIZED', 'FUTURE_PERIOD_YEAR_STD_MEDIAN_OF_MODELS_NORMALIZED', 'MODEL_CONFIDENCE_NORMALIZED', 'ANOMALY_MEAN_FUTURE_MINUS_REF_NORMALIZED', 'ANOMALY_STD_FUTURE_MINUS_REF_NORMALIZED', 'REFERENCE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_RAW_NORMALIZED', 'REFERENCE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_LABEL_NORMALIZED', 'FUTURE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_RAW_NORMALIZED', 'FUTURE_PERIOD_YEAR_ABSOLUTE_LEVEL_OF_EXPOSURE_LABEL_NORMALIZED', 'SITE_NAME_NORMALIZED', 'SITE_TYPOLOGY_NORMALIZED', 'STRU

STC_UP_IDENTIFIER,site_display_name,STC_UP_LABEL,STC_UP_ADDRESS_TOWN,STC_UP_COUNTRY_LABEL,STC_UP_GPS_LONGITUDE,STC_UP_GPS_LATITUDE,ecoact_site_id,ecoact_site_name,climate_latitude,climate_longitude,structure_code,structure_label,structure_activity,site_typology,climate_indicator,climate_hazard_family,eu_taxonomy_hazard,climate_scenario,reference_period,future_period,model_confidence,number_of_models,reference_mean,future_mean,climate_anomaly,reference_exposure_label,future_exposure_label,future_exposure_raw,climate_risk_score,is_fire_risk
null,null,null,null,null,null,null,38,CICO,42.55707,9.49079,UE2142,CNB,BPE,Carriere,cold_solifluction_chronic_trend,Soil,Solifluction,rcp85,1971-2000,2021-2040,high,7.0,21.0,10.9,-10.1,medium,medium,43.39,2,0
null,null,null,null,null,null,null,39,CICO,42.55707,9.49079,UE2156,SEP CNB,BPE,Carriere,cold_solifluction_chronic_trend,Soil,Solifluction,rcp85,1971-2000,2021-2040,high,7.0,21.0,10.9,-10.1,medium,medium,43.39,2,0
null,null,null,null,null,null,null,39,CICO,42.55707,9.49079,UE2156,SEP CNB,BPE,Carriere,wind_low_wind_acute_frequency,Wind,null,rcp26,1971-2000,2081-2100,low,6.0,265.8333333,266.9473684,1.114035088,high,high,89.66,3,0
null,null,null,null,null,null,null,39,CICO,42.55707,9.49079,UE2156,SEP CNB,BPE,Carriere,heat_heat_wave_threshold,Heat,Heat wave,rcp85,1971-2000,2081-2100,high,6.0,1.283333333,16.13157895,14.84824561,low,medium,47.94,2,0
UP3257,CNBSP- SITE DE BORGO,CNBSP- SITE DE BORGO,BORGO,France,9.44933032989502,42.55691146850586,40,CICO,42.55707,9.49079,UP3257,CNBSP- SITE DE BORGO,Blocs de Beton,Carriere,wind_tornadoes_acute_frequency,Wind,Tornado,historical,not applicable,not applicable,low,1.0,77.0,null,null,medium - high,medium - high,70.0,3,0
UP3257,CNBSP- SITE DE BORGO,CNBSP- SITE DE BORGO,BORGO,France,9.44933032989502,42.55691146850586,40,CICO,42.55707,9.49079,UP3257,CNBSP- SITE DE BORGO,Blocs de Beton,Carriere,wet_hail_acute_frequency,Storm / Hail,null,historical,1979-2015,not applicable,low,1.0,2.382198709,null,null,high,-999999,null,null,0
UP3257,CNBSP- SITE DE BORGO,CNBSP- SITE DE BORGO,BORGO,France,9.44933032989502,42.55691146850586,40,CICO,42.55707,9.49079,UP3257,CNBSP- SITE DE BORGO,Blocs de Beton,Carriere,water_saline_intrusion_trend,Water,Saline intrusion,rcp85,2015,2040,medium,1.0,0.219307495,0.381435006,0.16212751,medium - high,medium - high,78.14,3,0
UP3257,CNBSP- SITE DE BORGO,CNBSP- SITE DE BORGO,BORGO,France,9.44933032989502,42.55691146850586,40,CICO,42.55707,9.49079,UP3257,CNBSP- SITE DE BORGO,Blocs de Beton,Carriere,soil_soil_degradation_chronic_trend,Soil,Soil degradation,historical,2012,not applicable,low,1.0,null,null,null,-999999,-999999,null,null,0
UP891,CICO - BORGO,CICO - BORGO,BORGO,France,9.492650032043457,42.560791015625,41,CICO,42.55707,9.49079,UP891,CICO - BORGO,Blocs de Beton,Carriere,cooling_degree_days,Heat,null,rcp85,1971-2000,2041-2060,high,6.0,166.219094,304.9561345,138.7370405,medium,medium,47.75,2,0
UP891,CICO - BORGO,CICO - BORGO,BORGO,France,9.492650032043457,42.560791015625,42,CICO,42.55707,9.49079,UP891,CICO - BORGO,Carriere,Carriere,heat_extreme_heat_chronic,Heat,Heat stress,rcp85,1971-2000,2081-2100,high,6.0,17.32546637,21.27554267,3.950076296,-999999,extreme,100.0,5,0


STC_UP_IDENTIFIER,max_climate_risk_score,avg_climate_risk_score,nb_fire_risk_lines,nb_climate_indicators,nb_hazard_families,site_display_name,STC_UP_COUNTRY_LABEL,STC_UP_ADDRESS_TOWN,STC_UP_GPS_LONGITUDE,STC_UP_GPS_LATITUDE,climate_risk_level,has_fire_risk
UP825,5,2.287671232876712,12,44,7,CARRIERE D'AIGUES-VIVES,France,AIGUES- VIVES,4.197199821472168,43.70574951171875,Extreme,1
UP844,5,2.4275862068965517,24,44,7,USINE DE PREFAB DE MALIJAI,France,MALIJAI,6.031260013580322,44.04528045654297,Extreme,1
UP3257,5,2.270967741935484,12,44,7,CNBSP- SITE DE BORGO,France,BORGO,9.44933032989502,42.55691146850586,Extreme,1
UP743,5,2.384105960264901,12,44,7,CMM-SITE CARRIÈRE STE COLOMBE,France,THUIR,2.7400100231170654,42.62855911254883,Extreme,1
UP891,5,2.270967741935484,36,44,7,CICO - BORGO,France,BORGO,9.492650032043457,42.560791015625,Extreme,1
null,5,2.231832842518581,3060,44,7,null,null,null,null,null,Extreme,1
UP856,5,2.4275862068965517,12,44,7,null,null,null,null,null,Extreme,1
UP688,5,2.47972972972973,12,44,7,SITE DE LA GALIBERTE,France,BEZIERS,3.21274995803833,43.30747985839844,Extreme,1
UP4177,5,2.384105960264901,12,44,7,CARRIERE DE SAINTE COLOMBE,France,THUIR,2.7398500442504883,42.62776184082031,Extreme,1
UP820,5,2.47972972972973,36,44,7,null,null,null,null,null,Extreme,1


## Conversion multi-devises et reconstruction de la fiche carrière en EUR

Les données de livraison sont exprimées dans **12 devises différentes** (EUR, GBP, CHF, DKK, ZAR, NAD, XOF, XAF, XPF, MAD, MGA, MUR). Pour permettre une analyse consolidée, cette étape :

1. Crée une **table de taux de change** (`fx_rates_to_eur`) avec des taux fixes approximatifs
2. Normalise les codes devises non-standard (Dh → MAD, Ar → MGA, Rs → MUR)
3. Convertit les montants HT en EUR via jointure sur la table FX
4. Reconstruit la fiche carrière agrégée avec le CA exprimé en euros

**Limites connues** : les taux utilisés sont des approximations statiques ; pour une analyse financière précise, il faudrait des taux historiques par année.

**Table produite** : `fiche_carriere_bi_eur`

## Corrections de qualité sur les informations site

Ces cellules corrigent des problèmes de qualité identifiés dans `fiche_carriere_bi_eur` :

- **`site_status_bi`** : harmonisation du statut site ("Active" / "Unknown") pour éviter les valeurs incohérentes
- **`site_display_name_bi`** : fallback en cascade (`site_display_name` → `STC_UP_LABEL` → `STC_UP_IDENTIFIER`) pour garantir un nom affichable
- **`site_country_bi` / `site_town_bi` / `site_activity_bi`** : remplacement des NULL par "Unknown" pour éviter les trous dans les dashboards
- **`has_site_info`** / **`site_info_status`** : flag binaire indiquant si l'information site est présente

Ces corrections assurent que **100% des lignes ont un nom de site exploitable** pour les visualisations Power BI.

In [0]:
from pyspark.sql import functions as F

team_schema = "efrei_estp_2026.team6"

# ============================================================
# 1. Table de taux vers EUR
# A AJUSTER si vous avez de meilleurs taux historiques
# ============================================================

fx_data = [
    ("EUR", 1.0),
    ("GBP", 1.17),
    ("CHF", 1.05),
    ("DKK", 0.134),
    ("ZAR", 0.050),
    ("NAD", 0.050),
    ("XOF", 0.001524),
    ("XAF", 0.001524),
    ("XPF", 0.00838),
    ("MAD", 0.092),     # Dh = dirham marocain
    ("MGA", 0.00020),   # Ar = ariary malgache
    ("MUR", 0.021),     # Rs probable = roupie mauricienne, à vérifier
]

df_fx = spark.createDataFrame(fx_data, ["currency_clean", "rate_to_eur"])

df_fx.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{team_schema}.fx_rates_to_eur"
)

# ============================================================
# 2. Repartir de la table DLV nettoyée
# ============================================================

df_dlv = spark.table(f"{team_schema}.dlv_deliveries_f_clean")
df_sites = spark.table(f"{team_schema}.stc_sites_dim_clean")

# ============================================================
# 3. Normaliser les devises
# ============================================================

df_dlv = (
    df_dlv
    .withColumn(
        "currency_clean",
        F.regexp_replace(F.col("CUR_NAME"), "[()]", "")
    )
    .withColumn(
        "currency_clean",
        F.when(F.col("currency_clean") == "Dh", F.lit("MAD"))
         .when(F.col("currency_clean") == "Ar", F.lit("MGA"))
         .when(F.col("currency_clean") == "Rs", F.lit("MUR"))
         .otherwise(F.col("currency_clean"))
    )
)

# ============================================================
# 4. Joindre les taux FX
# ============================================================

df_dlv_fx = (
    df_dlv
    .join(df_fx, on="currency_clean", how="left")
    .withColumn(
        "DLV_PRE_TAX_AMOUNT_EUR",
        F.when(
            F.col("rate_to_eur").isNotNull(),
            F.col("DLV_PRE_TAX_AMOUNT") * F.col("rate_to_eur")
        )
    )
)

# Contrôle devises sans taux
display(
    df_dlv_fx
    .filter(F.col("currency_clean").isNotNull())
    .filter(F.col("rate_to_eur").isNull())
    .groupBy("CUR_NAME", "currency_clean")
    .count()
)

# ============================================================
# 5. Agréger : 1 ligne = 1 carrière + 1 année
# ============================================================

df_ca_eur = (
    df_dlv_fx
    .filter(F.col("STC_UP_IDENTIFIER").isNotNull())
    .filter(F.col("year").isNotNull())
    .groupBy("STC_UP_IDENTIFIER", "year")
    .agg(
        F.sum("DLV_PRE_TAX_AMOUNT").alias("ca_annuel_ht_local"),
        F.sum("DLV_PRE_TAX_AMOUNT_EUR").alias("ca_annuel_eur"),
        F.sum("DLV_QUANTITY").alias("quantite_totale_livree"),
        F.countDistinct("DLV_NUMBER_BL").alias("nb_bons_livraison"),
        F.countDistinct("TDP_CLIENT_MDM_IDENTIFIER").alias("nb_clients"),
        F.countDistinct("PRD_IDENTIFIER").alias("nb_produits"),
        F.countDistinct("currency_clean").alias("nb_devises"),
        F.collect_set("currency_clean").alias("devises_presentes")
    )
    .withColumn(
        "prix_moyen_unitaire_eur",
        F.when(
            F.col("quantite_totale_livree") > 0,
            F.col("ca_annuel_eur") / F.col("quantite_totale_livree")
        )
    )
)

# ============================================================
# 6. Ajouter infos site
# ============================================================

df_ca_eur = (
    df_ca_eur
    .join(
        df_sites.select(
            "STC_UP_IDENTIFIER",
            "site_display_name",
            "STC_UP_LABEL",
            "STC_UP_ADDRESS_TOWN",
            "STC_UP_COUNTRY_ID",
            "STC_UP_COUNTRY_LABEL",
            "STC_UP_GPS_LONGITUDE",
            "STC_UP_GPS_LATITUDE",
            "site_status",
            "STC_UP_ACTIVITY_1_LABEL_FR"
        ),
        on="STC_UP_IDENTIFIER",
        how="left"
    )
)

# ============================================================
# 7. Sauvegarder
# ============================================================

df_ca_eur.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{team_schema}.ca_annuel_par_carriere_eur"
)

print("Table créée : ca_annuel_par_carriere_eur")
print("Lignes :", df_ca_eur.count())

# Vérification doublons
display(
    df_ca_eur
    .groupBy("STC_UP_IDENTIFIER", "year")
    .count()
    .filter(F.col("count") > 1)
)

display(df_ca_eur)

CUR_NAME,currency_clean,count


Table créée : ca_annuel_par_carriere_eur
Lignes : 6858


STC_UP_IDENTIFIER,year,count


STC_UP_IDENTIFIER,year,ca_annuel_ht_local,ca_annuel_eur,quantite_totale_livree,nb_bons_livraison,nb_clients,nb_produits,nb_devises,devises_presentes,prix_moyen_unitaire_eur,site_display_name,STC_UP_LABEL,STC_UP_ADDRESS_TOWN,STC_UP_COUNTRY_ID,STC_UP_COUNTRY_LABEL,STC_UP_GPS_LONGITUDE,STC_UP_GPS_LATITUDE,site_status,STC_UP_ACTIVITY_1_LABEL_FR
UP2997,2022,974028.29,1139613.0993,47910.62999999999,1056,117,108,1,List(GBP),23.78622654930649,DEPOT EXETER,DEPOT EXETER,Exeter,GBR,Royaume-Uni,null,null,Active,null
UP1202,2025,5785460.559999998,5785460.559999998,2827589.040000001,26515,283,218,1,List(EUR),2.0460754650541424,SCPR SAINTE MARIE,SCPR SAINTE MARIE,SAINTE MARIE,FRA,France,55.5112190246582,-20.91408920288086,Active,null
UP2841,2019,189455.87999999995,189455.87999999995,300.25,73,2,2,1,List(EUR),630.9937718567859,CRAA AG MONTLUCON LE VEURDRE,CRAA AG MONTLUCON LE VEURDRE,LE VEURDRE,FRA,France,3.0406301021575928,46.75886917114258,Active,null
UP4869,2024,73269.59000000001,73269.59000000001,289.18,126,1,3,1,List(EUR),253.37018466007333,CNE VOSGES GEO,CNE VOSGES GEO,THAON-LES-VOSGES,FRA,France,null,null,Active,null
UP567,2020,6468353.869999999,null,2333615.1199999987,25014,80,19,0,List(),null,CARRIERE DE PERIGNY,CARRIERE DE PERIGNY,PERIGNY LA ROSE,FRA,France,3.6236701011657715,48.55691909790039,Active,"Carrières, gravières"
UP1151,2023,2104454.0499999993,2104454.0499999993,563364.54,10609,141,36,1,List(EUR),3.7355103145114517,CARRIERE DE SAINT JUST MALMONT,CARRIERE DE SAINT JUST MALMONT,SAINT-JUST MALMONT,FRA,France,4.336180210113525,45.34400177001953,Active,"Carrières, gravières"
UP548,2022,2661148.4199999995,2661148.4199999995,600452.2899999999,5059,34,26,1,List(EUR),4.4319065216655265,MOISSEY,MOISSEY,MOISSEY,FRA,France,5.523200035095215,47.19641876220703,Active,Dépôts pour négoce
UP1134,2019,520245.9000000001,520245.9000000001,73110.51999999999,38,5,8,1,List(EUR),7.115882912609569,ETABLISSEMENT DE LEMPDES,ETABLISSEMENT DE LEMPDES,LEMPDES,FRA,France,3.20920991897583,45.782291412353516,Active,null
UP3152,2021,135248.73,135248.73,1195.3200000000002,321,41,2,1,List(EUR),113.14855436201184,CRAA COMPOMAC CAR LA PATTE 69,CRAA COMPOMAC CAR LA PATTE 69,ST-LAURENT DE CHAMOUSSET,FRA,France,4.468430042266846,45.73944854736328,Active,null
UP606,2024,2976675.9399999995,2976675.9399999995,257270.14999999997,1235,114,61,1,List(EUR),11.570234401464763,CIDFN-COMPOMAC GENNEVILLIERS,CIDFN-COMPOMAC GENNEVILLIERS,GENNEVILLIERS,FRA,France,2.2985899448394775,48.946380615234375,Active,null


In [0]:
from pyspark.sql import functions as F

team_schema = "efrei_estp_2026.team6"
fiche_table = f"{team_schema}.fiche_carriere_bi_eur"

df = spark.table(fiche_table)

df_fixed = (
    df
    .withColumn(
        "site_status_bi",
        F.when(F.col("site_status") == "Active", F.lit("Active"))
         .otherwise(F.lit("Unknown"))
    )
    .withColumn(
        "has_site_info",
        F.when(F.col("site_display_name").isNotNull(), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "site_info_status",
        F.when(F.col("has_site_info") == 1, F.lit("Site info available"))
         .otherwise(F.lit("Missing site info"))
    )
)

df_fixed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fiche_table)

display(
    df_fixed
    .groupBy("site_status_bi", "site_info_status")
    .agg(
        F.count("*").alias("nb_lignes"),
        F.countDistinct("STC_UP_IDENTIFIER").alias("nb_sites"),
        F.sum("ca_annuel_eur").alias("ca_total_eur")
    )
)#

site_status_bi,site_info_status,nb_lignes,nb_sites,ca_total_eur
Active,Site info available,6357,1030,6.376310006914298E9
Unknown,Missing site info,501,142,1.456580747527001E8


In [0]:
from pyspark.sql import functions as F

team_schema = "efrei_estp_2026.team6"
fiche_table = f"{team_schema}.fiche_carriere_bi_eur"

df = spark.table(fiche_table)

df_fixed = (
    df
    .withColumn(
        "site_display_name_bi",
        F.coalesce(
            F.col("site_display_name"),
            F.col("STC_UP_LABEL"),
            F.col("STC_UP_IDENTIFIER")
        )
    )
    .withColumn(
        "site_country_bi",
        F.coalesce(
            F.col("STC_UP_COUNTRY_LABEL"),
            F.lit("Unknown")
        )
    )
    .withColumn(
        "site_town_bi",
        F.coalesce(
            F.col("STC_UP_ADDRESS_TOWN"),
            F.lit("Unknown")
        )
    )
    .withColumn(
        "site_activity_bi",
        F.coalesce(
            F.col("STC_UP_ACTIVITY_1_LABEL_FR"),
            F.lit("Unknown")
        )
    )
    .withColumn(
        "has_site_info",
        F.when(F.col("site_display_name").isNotNull(), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "site_info_status",
        F.when(F.col("has_site_info") == 1, F.lit("Site info available"))
         .otherwise(F.lit("Missing site info"))
    )
)

df_fixed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fiche_table)

display(
    df_fixed
    .groupBy("site_info_status")
    .agg(
        F.count("*").alias("nb_lignes"),
        F.countDistinct("STC_UP_IDENTIFIER").alias("nb_sites"),
        F.sum("ca_annuel_eur").alias("ca_total_eur")
    )
)

display(
    df_fixed
    .filter(F.col("site_info_status") == "Missing site info")
    .select(
        "STC_UP_IDENTIFIER",
        "site_display_name_bi",
        "year",
        "ca_annuel_eur",
        "quantite_totale_livree",
        "nb_clients"
    )
    .orderBy(F.desc("ca_annuel_eur"))
    .limit(50)
)

site_info_status,nb_lignes,nb_sites,ca_total_eur
Site info available,6357,1030,6.376310006914298E9
Missing site info,501,142,1.456580747527001E8


STC_UP_IDENTIFIER,site_display_name_bi,year,ca_annuel_eur,quantite_totale_livree,nb_clients
UP4226,UP4226,2022,1.758455501400001E7,72884.04000000002,158
UP4226,UP4226,2023,1.529847221759999E7,93652.01999999997,235
UP845,UP845,2021,1.077691139E7,992624.89,75
UP4395,UP4395,2022,9790491.870000005,804074.55,52
UP845,UP845,2022,9100397.420000002,912481.5199999996,54
UP845,UP845,2020,8267782.289999995,6.5214305899999976E7,71
UP845,UP845,2023,7332999.42,813246.0099999999,49
UP845,UP845,2019,6058990.7299999995,141642.28000000006,80
UP4268,UP4268,2022,5326521.549999998,275201.55,32
UP845,UP845,2018,3097497.63,32854.47000000001,56


In [0]:
df = spark.table("efrei_estp_2026.team6.fiche_carriere_bi_eur")
display(df.limit(20))

STC_UP_IDENTIFIER,year,site_display_name,STC_UP_LABEL,STC_UP_ADDRESS_TOWN,STC_UP_COUNTRY_ID,STC_UP_COUNTRY_LABEL,STC_UP_GPS_LONGITUDE,STC_UP_GPS_LATITUDE,site_status,STC_UP_ACTIVITY_1_LABEL_FR,ca_annuel_ht_local,ca_annuel_eur,quantite_totale_livree,nb_bons_livraison,nb_clients,nb_produits,nb_devises,devises_presentes,prix_moyen_unitaire_eur,production_volume,QYP_EXTRACTION_ANNUAL_FEES_REPORTING,extraction_cost_per_unit,is_active_production,reserve_volume,has_reserve_data,nb_breakdown_lines,total_breakdown_duration_hours,avg_breakdown_duration_hours,max_breakdown_duration_hours,reserve_life_years,availability_proxy,ca_per_client_eur,site_status_bi,has_site_info,site_info_status,site_display_name_bi,site_country_bi,site_town_bi,site_activity_bi
UP1218,2025,SITE DE LAROQUE D'OLMES,SITE DE LAROQUE D'OLMES,LAROQUES D OLMES,FRA,France,1.8653600215911865,42.96659851074219,Active,Décharges,23564.309999999998,23564.309999999998,3746.9999999999995,220,28,18,1,List(EUR),6.28884707766213,null,null,null,null,null,null,null,null,null,null,null,null,841.5824999999999,Active,1,Site info available,SITE DE LAROQUE D'OLMES,France,LAROQUES D OLMES,Décharges
UP2805,2022,CCO NED VANNES PLOEREN,CCO NED VANNES PLOEREN,PLOEREN,FRA,France,-2.867029905319214,47.65549850463867,Active,Dépôts pour négoce,234371.52,234371.52,39671.41,3308,68,41,1,List(EUR),5.907819258251722,null,null,null,null,null,null,null,null,null,null,null,null,3446.64,Active,1,Site info available,CCO NED VANNES PLOEREN,France,PLOEREN,Dépôts pour négoce
UP816,2022,SITE D'ESTRECHOUX,SITE D'ESTRECHOUX,ST-ETIENNE D ESTRECHOUX,FRA,France,3.10017991065979,43.66236877441406,Active,Dépôts pour négoce,134418.85,134418.85,23420.33,890,30,21,1,List(EUR),5.739408881087499,null,null,null,null,null,null,null,null,null,null,null,null,4480.628333333333,Active,1,Site info available,SITE D'ESTRECHOUX,France,ST-ETIENNE D ESTRECHOUX,Dépôts pour négoce
UP4167,2026,CARRIERE DE DOVILLE,CARRIERE DE DOVILLE,DOVILLE,FRA,France,-1.5654799938201904,49.333038330078125,Active,Centrale de grave traitée hydraulique,1192546.1200000006,1192546.1200000006,312371.63,3678,200,121,1,List(EUR),3.81771584058386,null,null,null,null,null,null,null,null,null,null,null,null,5962.730600000003,Active,1,Site info available,CARRIERE DE DOVILLE,France,DOVILLE,Centrale de grave traitée hydraulique
UP614,2022,null,null,null,null,null,null,null,null,null,770.02,null,118.94999999999999,1,1,5,0,List(),null,null,null,null,null,null,null,null,null,null,null,null,null,null,Unknown,0,Missing site info,UP614,Unknown,Unknown,Unknown
UP3158,2023,CRAA COMPOMAC ST DENIS FAMY 01,CRAA COMPOMAC ST DENIS FAMY 01,SAINT-DENIS-LES-BOURG,FRA,France,5.210149765014648,46.20962142944336,Active,null,45876.96,45876.96,341.04999999999995,144,20,3,1,List(EUR),134.51681571617067,null,null,null,null,null,null,null,null,null,null,null,null,2293.848,Active,1,Site info available,CRAA COMPOMAC ST DENIS FAMY 01,France,SAINT-DENIS-LES-BOURG,Unknown
UP1308,2018,null,null,null,null,null,null,null,null,null,4195187.570000003,null,913388.7299999996,18145,405,95,0,List(),null,null,null,null,null,null,null,null,null,null,null,null,null,null,Unknown,0,Missing site info,UP1308,Unknown,Unknown,Unknown
UP683,2025,CARRIERE DE CARLENCAS,CARRIERE DE CARLENCAS,CARLENCAS ET LEVAS,FRA,France,3.224220037460327,43.63230895996094,Active,null,14228.600000000002,14228.600000000002,2460.79,28,6,8,1,List(EUR),5.782126877953829,3344.00,0.00,0E-13,1,0.0,1,null,null,null,null,0.0,null,2371.433333333334,Active,1,Site info available,CARRIERE DE CARLENCAS,France,CARLENCAS ET LEVAS,Unknown
UP580,2019,REIMS ENR,REIMS ENR,REIMS,FRA,France,4.064789772033691,49.270320892333984,Active,Dépôts pour négoce,40952.409999999996,null,1504.77,225,11,8,0,List(),null,null,null,null,null,null,null,null,null,null,null,null,null,null,Active,1,Site info available,REIMS ENR,France,REIMS,Dépôts pour négoce
UP1202,2025,SCPR SAINTE MARIE,SCPR SAINTE MARIE,SAINTE MARIE,FRA,France,55.5112190246582,-20.91408920288086,Active

## Construction de la table résumée par carrière (`carriere_summary_bi`)

Cette étape agrège toutes les données au **grain carrière** (1 ligne = 1 site) pour produire une vue synthétique 360°.

**Dimensions agrégées :**
- Période d'activité (première/dernière année, nombre d'années)
- Performance financière (CA total, moyen, max en EUR)
- Volumes (production cumulée, quantités livrées)
- Clientèle (nombre total de clients, revenu moyen par client)
- Réserves (volume max, durée de vie estimée, ratio réserve/production)
- Maintenance (pannes cumulées, durée totale, proxy disponibilité)
- Risques climatiques (score max, score moyen, niveau de risque)

**Profil business** : chaque site reçoit une catégorie stratégique :
- *High revenue / climate risk* — CA > 10M€ et score climat ≥ 3
- *High revenue / operational risk* — CA > 10M€ et disponibilité < 95%
- *Reserve pressure* — ratio réserve/production < 5 ans
- *Strategic high revenue* — CA > 10M€ sans risque identifié
- *Standard monitoring* — autres

**Table produite** : `carriere_summary_bi`

In [0]:
from pyspark.sql import functions as F

team_schema = "efrei_estp_2026.team6"

input_fiche = f"{team_schema}.fiche_carriere_bi_eur"
input_climate = f"{team_schema}.climate_risks_site_summary_bi"
output_table = f"{team_schema}.carriere_summary_bi"

# ============================================================
# 1. Charger les tables
# ============================================================

df = spark.table(input_fiche)
df_climate = spark.table(input_climate)

# ============================================================
# 2. Agréger fiche_carriere_bi_eur
# Grain final : 1 ligne = 1 carrière
# ============================================================

df_summary = (
    df
    .groupBy("STC_UP_IDENTIFIER")
    .agg(
        # Infos site
        F.first("site_display_name_bi", ignorenulls=True).alias("site_display_name"),
        F.first("site_country_bi", ignorenulls=True).alias("country"),
        F.first("site_town_bi", ignorenulls=True).alias("town"),
        F.first("site_activity_bi", ignorenulls=True).alias("activity"),
        F.first("site_info_status", ignorenulls=True).alias("site_info_status"),
        F.first("site_status_bi", ignorenulls=True).alias("site_status_bi"),
        F.first("STC_UP_GPS_LONGITUDE", ignorenulls=True).alias("longitude"),
        F.first("STC_UP_GPS_LATITUDE", ignorenulls=True).alias("latitude"),

        # Période
        F.min("year").alias("first_year"),
        F.max("year").alias("last_year"),
        F.countDistinct("year").alias("nb_years"),

        # CA / ventes
        F.sum("ca_annuel_eur").alias("ca_total_eur"),
        F.avg("ca_annuel_eur").alias("ca_avg_yearly_eur"),
        F.max("ca_annuel_eur").alias("ca_max_yearly_eur"),
        F.sum("ca_annuel_ht_local").alias("ca_total_local"),
        F.sum("quantite_totale_livree").alias("quantity_total"),
        F.avg("prix_moyen_unitaire_eur").alias("avg_unit_price_eur"),
        F.sum("nb_bons_livraison").alias("nb_delivery_notes_total"),
        F.sum("nb_clients").alias("nb_clients_total"),
        F.sum("nb_produits").alias("nb_products_total"),
        F.max("nb_devises").alias("max_nb_currencies"),

        # Production / réserves
        F.sum("production_volume").alias("production_total"),
        F.avg("production_volume").alias("production_avg_yearly"),
        F.max("reserve_volume").alias("reserve_volume_latest_or_max"),
        F.avg("reserve_life_years").alias("avg_reserve_life_years"),
        F.max("reserve_life_years").alias("max_reserve_life_years"),

        # Pannes
        F.sum("nb_breakdown_lines").alias("nb_breakdowns_total"),
        F.sum("total_breakdown_duration_hours").alias("breakdown_duration_total_hours"),
        F.avg("availability_proxy").alias("avg_availability_proxy"),

        # Qualité conversion
        F.count(F.when(F.col("ca_annuel_eur").isNull() & F.col("ca_annuel_ht_local").isNotNull(), 1)).alias("nb_years_revenue_not_converted"),
        F.count(F.when(F.col("ca_annuel_eur").isNotNull(), 1)).alias("nb_years_revenue_converted")
    )
)

# ============================================================
# 3. Ajouter des KPI calculés
# ============================================================

df_summary = (
    df_summary
    .withColumn(
        "avg_revenue_per_client_eur",
        F.when(F.col("nb_clients_total") > 0, F.col("ca_total_eur") / F.col("nb_clients_total"))
    )
    .withColumn(
        "avg_revenue_per_quantity_eur",
        F.when(F.col("quantity_total") > 0, F.col("ca_total_eur") / F.col("quantity_total"))
    )
    .withColumn(
        "reserve_to_production_ratio",
        F.when(F.col("production_total") > 0, F.col("reserve_volume_latest_or_max") / F.col("production_total"))
    )
    .withColumn(
        "has_revenue_conversion_issue",
        F.when(F.col("nb_years_revenue_not_converted") > 0, F.lit(1)).otherwise(F.lit(0))
    )
)

# ============================================================
# 4. Ajouter les risques climatiques
# ============================================================

df_climate_one = (
    df_climate
    .groupBy("STC_UP_IDENTIFIER")
    .agg(
        F.max("max_climate_risk_score").alias("max_climate_risk_score"),
        F.avg("avg_climate_risk_score").alias("avg_climate_risk_score"),
        F.max("nb_fire_risk_lines").alias("nb_fire_risk_lines"),
        F.max("nb_climate_indicators").alias("nb_climate_indicators"),
        F.max("nb_hazard_families").alias("nb_hazard_families"),
        F.first("climate_risk_level", ignorenulls=True).alias("climate_risk_level"),
        F.max("has_fire_risk").alias("has_fire_risk")
    )
)

df_summary = (
    df_summary
    .join(df_climate_one, on="STC_UP_IDENTIFIER", how="left")
)

# ============================================================
# 5. Créer une priorité business simple
# ============================================================

df_summary = (
    df_summary
    .withColumn(
        "business_profile",
        F.when(
            (F.col("ca_total_eur") > 10000000) & (F.col("max_climate_risk_score") >= 3),
            F.lit("High revenue / climate risk")
        )
        .when(
            (F.col("ca_total_eur") > 10000000) & (F.col("avg_availability_proxy") < 0.95),
            F.lit("High revenue / operational risk")
        )
        .when(
            (F.col("reserve_to_production_ratio") < 5) & F.col("reserve_to_production_ratio").isNotNull(),
            F.lit("Reserve pressure")
        )
        .when(
            F.col("ca_total_eur") > 10000000,
            F.lit("Strategic high revenue")
        )
        .otherwise(F.lit("Standard monitoring"))
    )
)

# ============================================================
# 6. Sauvegarder
# ============================================================

df_summary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)

print(f"Table créée : {output_table}")
print("Lignes :", df_summary.count())
print("Carrières distinctes :", df_summary.select("STC_UP_IDENTIFIER").distinct().count())

# Vérification : aucun doublon attendu
display(
    df_summary
    .groupBy("STC_UP_IDENTIFIER")
    .count()
    .filter(F.col("count") > 1)
)

display(df_summary)

Table créée : efrei_estp_2026.team6.carriere_summary_bi
Lignes : 1172
Carrières distinctes : 1172


STC_UP_IDENTIFIER,count


STC_UP_IDENTIFIER,site_display_name,country,town,activity,site_info_status,site_status_bi,longitude,latitude,first_year,last_year,nb_years,ca_total_eur,ca_avg_yearly_eur,ca_max_yearly_eur,ca_total_local,quantity_total,avg_unit_price_eur,nb_delivery_notes_total,nb_clients_total,nb_products_total,max_nb_currencies,production_total,production_avg_yearly,reserve_volume_latest_or_max,avg_reserve_life_years,max_reserve_life_years,nb_breakdowns_total,breakdown_duration_total_hours,avg_availability_proxy,nb_years_revenue_not_converted,nb_years_revenue_converted,avg_revenue_per_client_eur,avg_revenue_per_quantity_eur,reserve_to_production_ratio,has_revenue_conversion_issue,max_climate_risk_score,avg_climate_risk_score,nb_fire_risk_lines,nb_climate_indicators,nb_hazard_families,climate_risk_level,has_fire_risk,business_profile
UP1308,UP1308,Unknown,Unknown,Unknown,Missing site info,Unknown,null,null,2015,2020,6,null,null,null,2.1845309340000007E7,4685409.879999996,null,92105,2244,527,0,null,null,null,null,null,null,null,null,6,0,null,null,null,1,null,null,null,null,null,null,null,Standard monitoring
UP917,CARRIERE DE FAYENCE,France,FAYENCE,Décharges,Site info available,Active,6.6935200691223145,43.64025115966797,2016,2024,9,882989.28,294329.76,657957.85,5303802.069999999,538788.5,9.180904291705218,18444,454,632,1,26208.00,8736.000000,0.0,0.0,0.0,null,null,null,6,3,1944.910308370044,1.6388421059469533,0.0,1,5,2.2,24,44,7,Extreme,1,Reserve pressure
UP615,TERSEN-NEGOCE DIRECTE SMS,France,MARCOUSSIS,Unknown,Site info available,Active,2.196160078048706,48.66791915893555,2017,2026,7,2287559.29,762519.7633333333,1301381.4400000002,3211276.9600000004,280002.1999999999,14.207030150499087,5775,317,160,1,null,null,null,null,null,null,null,null,4,3,7216.275362776026,8.16979041593245,null,1,5,2.191780821917808,12,44,7,Extreme,1,Standard monitoring
UP531,CARRIERE DE VERGRANNE,France,VERGRANNE,"Carrières, gravières",Site info available,Active,6.388629913330078,47.40919876098633,2019,2026,8,3115224.37,389403.04625,688757.7300000002,3115224.37,1490323.0999999999,2.162908545486557,29245,589,416,1,181091.00,45272.750000,0.0,0.0,0.0,null,null,null,0,8,5289.00572156197,2.0903013380118716,0.0,0,5,2.075187969924812,12,44,7,Extreme,1,Reserve pressure
UP3173,CRAA AG MONTCEAU COMPO ETANG,France,ETANG SUR ARROUX,Unknown,Site info available,Active,4.176869869232178,46.86188888549805,2019,2023,5,131504.12,26300.824,44043.62,131504.12,1286.89,100.85152012966073,38,5,10,1,null,null,null,null,null,null,null,null,0,5,26300.824,102.18753739635865,null,0,null,null,null,null,null,null,null,Standard monitoring
UP4047,CARRIERE DE GLENIC,France,GLENIC,"Carrières, gravières",Site info available,Active,1.9181300401687622,46.21773147583008,2021,2026,6,4132221.03,688703.505,1291761.4400000002,4132221.03,1099096.4599999997,4.704679162493964,27179,600,234,1,97160.00,24290.000000,0.0,0.0,0.0,null,null,null,0,6,6887.0350499999995,3.7596527514973532,0.0,0,5,2.227272727272727,12,44,7,Extreme,1,Reserve pressure
UP3722,CSO AG NEGOCE DORDOGNE,France,SAINT ASTIER,Dépôts pour négoce,Site info available,Active,0.5543199777603149,45.15010070800781,2019,2026,8,294643.29000000004,36830.411250000005,97765.13,294643.29000000004,5416.24,67.9339274094803,406,95,58,1,null,null,null,null,null,null,null,null,0,8,3101.508315789474,54.399969351432,null,0,null,null,null,null,null,null,null,Standard monitoring
UP4075,CNE ARDENNES EMUL SEDAN,France,SEDAN,Unknown,Site info available,Active,4.919829845428467,49.698089599609375,2021,2025,5,800763.62,160152.724,237915.54,800763.62,5872.64,574.7761640801937,680,14,7,1,null,null,null,null,null,null,null,null,0,5,57197.40142857143,136.35496471774192,null,0,null,null,null,null,null,null,null,Standard monitoring
UP3087,CMA EL GARA,Maroc,BENSLIMANE,Unknown,Site info available,Active,-7.118720054626465,33.61302947998047,2021,2026,6,null,null,null,8.021743359E7,1942515.04,null,27679,96,90,1,939724.37,313241.456667,0.0,0.0,0.0,null,null,null,6,0,null,null,0.0,1,null,n

## Enrichissement du résumé climatique (V2 — score composite)

La V1 du résumé climatique (`climate_risks_site_summary_bi`) utilisait uniquement le **score max** par site, ce qui manquait de discrimination (la majorité des sites avaient un score max élevé).

Cette V2 introduit :

1. **Score composite pondéré** = 50% score moyen + 20% score max + 30% part des risques élevés
2. **Catégorisation par quantiles** (20/40/60/80) pour obtenir 5 niveaux relatifs : Low, Limited, Moderate, High, Very high
3. **Famille de risque dominante** par site (la famille avec le score moyen le plus élevé)
4. Conservation de l'ancien niveau (basé sur le max) pour comparaison

Ce score plus discriminant permet de mieux prioriser les sites dans les dashboards.

**Table overwrite** : `climate_risks_site_summary_bi`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

team_schema = "efrei_estp_2026.team6"

input_table = f"{team_schema}.climate_risks_bi"
output_table = f"{team_schema}.climate_risks_site_summary_bi"  # on overwrite la V1

df = spark.table(input_table)

# ============================================================
# 1. Résumé climat par carrière
# ============================================================

df_summary = (
    df
    .groupBy("STC_UP_IDENTIFIER")
    .agg(
        F.first("site_display_name", ignorenulls=True).alias("site_display_name"),
        F.first("STC_UP_COUNTRY_LABEL", ignorenulls=True).alias("country"),
        F.first("STC_UP_ADDRESS_TOWN", ignorenulls=True).alias("town"),
        F.first("STC_UP_GPS_LONGITUDE", ignorenulls=True).alias("longitude"),
        F.first("STC_UP_GPS_LATITUDE", ignorenulls=True).alias("latitude"),

        F.max("climate_risk_score").alias("max_climate_risk_score"),
        F.avg("climate_risk_score").alias("avg_climate_risk_score"),
        F.count("*").alias("nb_climate_lines"),
        F.countDistinct("climate_indicator").alias("nb_climate_indicators"),
        F.countDistinct("climate_hazard_family").alias("nb_hazard_families"),

        F.sum(F.when(F.col("climate_risk_score") == 5, 1).otherwise(0)).alias("nb_extreme_risks"),
        F.sum(F.when(F.col("climate_risk_score") == 4, 1).otherwise(0)).alias("nb_very_high_risks"),
        F.sum(F.when(F.col("climate_risk_score") == 3, 1).otherwise(0)).alias("nb_high_risks"),
        F.sum(F.when(F.col("climate_risk_score") >= 3, 1).otherwise(0)).alias("nb_high_or_extreme_risks"),

        F.max("is_fire_risk").alias("has_fire_risk"),
        F.sum("is_fire_risk").alias("nb_fire_risk_lines")
    )
    .withColumn(
        "share_high_or_extreme",
        F.when(
            F.col("nb_climate_lines") > 0,
            F.col("nb_high_or_extreme_risks") / F.col("nb_climate_lines")
        )
    )
)

# ============================================================
# 2. Score climat composite plus discriminant
# ============================================================

df_summary = (
    df_summary
    .withColumn(
        "climate_composite_score",
        (
            F.coalesce(F.col("avg_climate_risk_score"), F.lit(0)) * 0.50
            + F.coalesce(F.col("max_climate_risk_score"), F.lit(0)) * 0.20
            + F.coalesce(F.col("share_high_or_extreme"), F.lit(0)) * 5 * 0.30
        )
    )
)

# ============================================================
# 3. Catégories relatives par quantiles
# ============================================================

quantiles = df_summary.approxQuantile(
    "climate_composite_score",
    [0.20, 0.40, 0.60, 0.80],
    0.01
)

q20, q40, q60, q80 = quantiles

df_summary = df_summary.withColumn(
    "climate_risk_level_bi",
    F.when(F.col("climate_composite_score") >= q80, F.lit("Very high"))
     .when(F.col("climate_composite_score") >= q60, F.lit("High"))
     .when(F.col("climate_composite_score") >= q40, F.lit("Moderate"))
     .when(F.col("climate_composite_score") >= q20, F.lit("Limited"))
     .otherwise(F.lit("Low"))
)

# ============================================================
# 4. Garder aussi l'ancien niveau basé sur le max
# ============================================================

df_summary = df_summary.withColumn(
    "climate_risk_level",
    F.when(F.col("max_climate_risk_score") >= 5, F.lit("Extreme"))
     .when(F.col("max_climate_risk_score") >= 4, F.lit("Very high"))
     .when(F.col("max_climate_risk_score") >= 3, F.lit("High"))
     .when(F.col("max_climate_risk_score") >= 2, F.lit("Medium"))
     .when(F.col("max_climate_risk_score") >= 1, F.lit("Low"))
     .otherwise(F.lit("Unknown"))
)

# ============================================================
# 5. Famille de risque dominante par site
# ============================================================

df_family = (
    df
    .groupBy("STC_UP_IDENTIFIER", "climate_hazard_family")
    .agg(
        F.avg("climate_risk_score").alias("family_avg_risk_score"),
        F.count("*").alias("family_nb_lines")
    )
)

w = Window.partitionBy("STC_UP_IDENTIFIER").orderBy(
    F.desc("family_avg_risk_score"),
    F.desc("family_nb_lines")
)

df_family_top = (
    df_family
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select(
        "STC_UP_IDENTIFIER",
        F.col("climate_hazard_family").alias("dominant_hazard_family"),
        "family_avg_risk_score"
    )
)

df_summary = df_summary.join(df_family_top, on="STC_UP_IDENTIFIER", how="left")

# ============================================================
# 6. Sauvegarder en écrasant la V1
#c2kgSUE=Y2UgZGV2b2lyIHJlc3BlY3RlIHRvdXMgbGVzIGF0dGVuZHVz
# ============================================================

df_summary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)

print(f"Table overwritée : {output_table}")
print("Lignes :", df_summary.count())
print("Sites distincts :", df_summary.select("STC_UP_IDENTIFIER").distinct().count())

print("Distribution ancien niveau max :")
display(
    df_summary.groupBy("climate_risk_level")
    .agg(F.count("*").alias("nb_sites"))
    .orderBy(F.desc("nb_sites"))
)

print("Distribution nouveau niveau BI :")
display(
    df_summary.groupBy("climate_risk_level_bi")
    .agg(
        F.count("*").alias("nb_sites"),
        F.avg("climate_composite_score").alias("avg_score")
    )
    .orderBy("avg_score")
)

display(df_summary)

Table overwritée : efrei_estp_2026.team6.climate_risks_site_summary_bi
Lignes : 253
Sites distincts : 253
Distribution ancien niveau max :


climate_risk_level,nb_sites
Extreme,253


Distribution nouveau niveau BI :


climate_risk_level_bi,nb_sites,avg_score
Low,49,2.255202150795222
Limited,51,2.376617468140694
Moderate,50,2.45206157892838
High,50,2.511210925453986
Very high,53,2.6245725301458322


STC_UP_IDENTIFIER,site_display_name,country,town,longitude,latitude,max_climate_risk_score,avg_climate_risk_score,nb_climate_lines,nb_climate_indicators,nb_hazard_families,nb_extreme_risks,nb_very_high_risks,nb_high_risks,nb_high_or_extreme_risks,has_fire_risk,nb_fire_risk_lines,share_high_or_extreme,climate_composite_score,climate_risk_level_bi,climate_risk_level,dominant_hazard_family,family_avg_risk_score
UP996,CARRIERE DE CIRES LES MELLO,France,CIRES LES MELLO,2.3296000957489014,49.24909973144531,5,2.117241379310345,219,44,7,7,0,42,49,1,12,0.2237442922374429,2.3942371280113366,Limited,Extreme,Other,2.6153846153846154
UP3045,SITE DE NESMY,France,NESMY,-1.4328800439834595,46.59403991699219,5,2.0965517241379312,219,44,7,11,0,26,37,1,12,0.1689497716894977,2.3017005196032123,Low,Extreme,Heat,2.388888888888889
UP615,TERSEN-NEGOCE DIRECTE SMS,France,MARCOUSSIS,2.196160078048706,48.66791915893555,5,2.191780821917808,219,44,7,8,0,45,53,1,12,0.2420091324200913,2.458904109589041,Moderate,Extreme,Other,2.6153846153846154
UP1408,null,null,null,null,null,5,2.213793103448276,219,44,7,11,0,39,50,1,12,0.228310502283105,2.449362305148796,Moderate,Extreme,Other,2.6153846153846154
UP531,CARRIERE DE VERGRANNE,France,VERGRANNE,6.388629913330078,47.40919876098633,5,2.075187969924812,219,44,7,2,0,42,44,1,12,0.2009132420091324,2.3389638479761046,Limited,Extreme,Other,2.769230769230769
UP590,CARRIERE DE TRAPP,France,RAON L ETAPE,6.88640022277832,48.41714859008789,5,2.0902255639097747,219,44,7,2,0,44,46,1,12,0.2100456621004566,2.360181275105572,Limited,Extreme,Other,2.769230769230769
UP1209,SCPR PIERREFONDS ST PIERRE,France,SAINT PIERRE,55.421119689941406,-21.29737091064453,5,1.950354609929078,438,44,7,20,0,60,80,1,24,0.182648401826484,2.2491499077042647,Low,Extreme,Drought,2.5
UP1153,null,null,null,null,null,5,2.2761194029850746,219,44,7,18,0,32,50,1,12,0.228310502283105,2.4805254549171947,High,Extreme,Other,3.5714285714285716
UP3269,CONC MOBILE PY,France,VELESMES ESSARTS,5.8724799156188965,47.196678161621094,5,2.074626865671642,438,44,7,8,0,78,86,1,24,0.1963470319634703,2.331833980781026,Limited,Extreme,Other,2.769230769230769
UP769,CMM-SITE DE PONT DE GUEYDAN,France,ANNOT,6.672560214996338,43.961761474609375,5,2.2482758620689656,876,44,7,56,0,172,228,1,48,0.2602739726027397,2.514548889938592,High,Extreme,Other,3.6153846153846154


In [0]:
from pyspark.sql import functions as F

df = spark.table("efrei_estp_2026.team6.qib_installations_breakdowns_monthly_kpi")

display(
    df.groupBy("year")
    .agg(
        F.count("*").alias("nb_lignes"),
        F.sum("nb_breakdown_lines").alias("total_pannes")
    )
    .orderBy("year")
)

year,nb_lignes,total_pannes
2024,1,1
2025,271,2185
2026,78,671


In [0]:
from pyspark.sql import functions as F

df = spark.table("efrei_estp_2026.team6.carriere_summary_bi")

display(
    df.select(
        "STC_UP_IDENTIFIER",
        "site_display_name",
        "production_total",
        "reserve_volume_latest_or_max",
        "reserve_to_production_ratio"
    )
    .orderBy(F.desc("reserve_volume_latest_or_max"))
    .limit(30)
)

display(
    df.agg(
        F.count("*").alias("nb_carrieres"),
        F.count(F.when(F.col("production_total").isNotNull(), 1)).alias("avec_production"),
        F.count(F.when(F.col("reserve_volume_latest_or_max").isNotNull(), 1)).alias("avec_reserves"),
        F.count(F.when(F.col("reserve_volume_latest_or_max") > 0, 1)).alias("avec_reserves_positives"),
        F.count(F.when(F.col("reserve_to_production_ratio") < 5, 1)).alias("sous_pression_reserve")
    )
)

STC_UP_IDENTIFIER,site_display_name,production_total,reserve_volume_latest_or_max,reserve_to_production_ratio
UP551,CARRIERE DE RUBECOURT,719686.00,3938350.0,5.472317093843704
UP590,CARRIERE DE TRAPP,5413707.00,2043000.0,0.37737542870347435
UP1334,SCMC BAGNAC,795862.00,1519666.0,1.9094591775961158
UP1353,CARRIERE DE ST-MARTIN D'ONEY,1044438.00,997147.0,0.9547211035982988
UP548,MOISSEY,475108.00,825000.0,1.7364472919841383
UP1343,CARRIERE DE SAINT GERME,1119520.00,775010.0,0.6922699013863084
UP3214,CARRIERE DE JALONS,519757.00,684828.0,1.3175926442549115
UP4014,CARRIERE DE TREGLAMUS,290654.00,673000.0,2.3154678758936744
UP5182,CARRIERE MORGAGNI VAUCLERC,40121.00,611265.0,15.235537499065327
UP524,CCM WALLERS EN FAGNE,4668643.00,579000.0,0.1240189065644985


nb_carrieres,avec_production,avec_reserves,avec_reserves_positives,sous_pression_reserve
1172,246,223,63,185


# Conclusion — Tables produites et architecture

Ce pipeline Data Engineering transforme **7 sources brutes** en **10 tables analytiques** exploitées par les dashboards Power BI :

| Table | Grain | Objectif |
| --- | --- | --- |
| `dlv_deliveries_f_clean` | 1 livraison | Données de livraison nettoyées |
| `qib_installations_breakdowns_monthly_kpi` | site × mois | KPIs de pannes et disponibilité |
| `prd_products_dim_clean` | 1 produit | Référentiel produits nettoyé |
| `stc_sites_dim_clean` | 1 site | Dimension carrières (géo, statut, activité) |
| `ecoact_data_clean` | 1 indicateur climat | Données climatiques EcoAct nettoyées |
| `qyp_quarries_yearly_kpi` | site × année | KPIs de production annuelle |
| `ryp_quarries_reserves_yearly_kpi` | site × année | KPIs de réserves |
| `fiche_carriere_bi_eur` | site × année | Fiche complète avec CA en EUR |
| `carriere_summary_bi` | 1 site | Vue 360° agrégée + profil business |
| `climate_risks_site_summary_bi` | 1 site | Score climatique composite + catégorisation |

**Architecture** : Source brute → Nettoyage → KPIs intermédiaires → Tables analytiques finales → Power BI

**Points clés** :
- Conversion multi-devises vers EUR pour consolidation financière
- Scoring multicritères (Min-Max normalisé par année)
- Catégorisation stratégique automatique des carrières
- Score climatique composite discriminant (quantiles)
- Fallbacks systématiques pour garantir la complétude des dashboards